In [1]:
import importlib
importlib.reload(importlib.import_module('analysis_utils_msr'))
import warnings
import pandas as pd
import numpy as np
from analysis_utils_msr import sum_individual_mutation_scores, assess_grouped_spearman, assess_grouped_gain, assess_grouped_ndcg, assess_classification_metric, assess_all_models, add_additive_predictions, unify_similar_columns, deduplicate_and_check, compute_dddg
import pandas as pd
import numpy as np
from functools import partial

# notebook display options
warnings.filterwarnings('ignore')

#hybrid_base = f'{base_path}/msr_dual_ensemble_masked_ss12/1/epoch=03-val_rho_combined_avg=0.818.ckpt'

high_alpha = '20.0'
full_alpha = '16.0'
med_alpha = '8.0'
low_alpha = '4.0'

In [2]:
results = {}

for dataset in ['domainome', 'hyperopt_splits-test', 'hyperopt_splits-val']:
    print(dataset)

    base_path = f'~/software/esm-msr/analysis_notebooks/predictions/{dataset}'
    msr_base_1 = f'{base_path}/msr_dual_ensemble_small_large/1/epoch=05-val_rho_combined_avg=0.805.ckpt'
    msr_base_2 = f'{base_path}/msr_dual_ensemble_small_large/2/epoch=04-val_rho_combined_avg=0.809.ckpt'
    msr_base_3 = f'{base_path}/msr_dual_ensemble_small_large/3/epoch=03-val_rho_combined_avg=0.807.ckpt'

    chain_base_1 = f'{base_path}/msr_dual_ensemble_small_large_chain/1/epoch=04-val_rho_combined_avg=0.818.ckpt'
    chain_base_2 = f'{base_path}/msr_dual_ensemble_small_large_chain/2/epoch=01-val_rho_combined_avg=0.816.ckpt'
    chain_base_3 = f'{base_path}/msr_dual_ensemble_small_large_chain/3/epoch=02-val_rho_combined_avg=0.817.ckpt'

    mm_base_1 = f'{base_path}/msr_dual_ensemble_small_large_mm/1/epoch=02-val_rho_combined_avg=0.808.ckpt'
    mm_base_2 = f'{base_path}/msr_dual_ensemble_small_large_mm/2/epoch=02-val_rho_combined_avg=0.811.ckpt'
    mm_base_3 = f'{base_path}/msr_dual_ensemble_small_large_mm/3/epoch=02-val_rho_combined_avg=0.807.ckpt'

    single_base_1 = f'{base_path}/msr_dual_ensemble_small_large_singles/1/epoch=05-val_rho_combined_avg=0.809.ckpt'
    single_base_2 = f'{base_path}/msr_dual_ensemble_small_large_singles/2/epoch=05-val_rho_combined_avg=0.810.ckpt'
    single_base_3 = f'{base_path}/msr_dual_ensemble_small_large_singles/3/epoch=03-val_rho_combined_avg=0.809.ckpt'

    reg_base_1 = f'{base_path}/msr_dual_ensemble_small_large_norank_mse20/1/epoch=03-val_rho_combined_avg=0.809.ckpt'
    reg_base_2 = f'{base_path}/msr_dual_ensemble_small_large_norank_mse20/2/epoch=06-val_rho_combined_avg=0.812.ckpt'
    reg_base_3 = f'{base_path}/msr_dual_ensemble_small_large_norank_mse20/3/epoch=06-val_rho_combined_avg=0.812.ckpt'

    noseqhead_base_1 = f'{base_path}/msr_dual_ensemble_small_large_noseqhead/1/epoch=04-val_rho_combined_avg=0.821.ckpt'
    noseqhead_base_2 = f'{base_path}/msr_dual_ensemble_small_large_noseqhead/2/epoch=08-val_rho_combined_avg=0.821.ckpt'
    noseqhead_base_3 = f'{base_path}/msr_dual_ensemble_small_large_noseqhead/3/epoch=03-val_rho_combined_avg=0.820.ckpt'

    all_base_1 = f'{base_path}/msr_dual_ensemble_small_large_all/1/epoch=02-val_rho_combined_avg=0.821.ckpt'
    all_base_2 = f'{base_path}/msr_dual_ensemble_small_large_all/2/epoch=02-val_rho_combined_avg=0.821.ckpt'
    all_base_3 = f'{base_path}/msr_dual_ensemble_small_large_all/3/epoch=03-val_rho_combined_avg=0.823.ckpt'

    ffn_base_1 = f'{base_path}/msr_dual_ensemble_small_large_ffn/1/epoch=06-val_rho_combined_avg=0.816.ckpt'
    ffn_base_2 = f'{base_path}/msr_dual_ensemble_small_large_ffn/2/epoch=04-val_rho_combined_avg=0.820.ckpt'
    ffn_base_3 = f'{base_path}/msr_dual_ensemble_small_large_ffn/3/epoch=03-val_rho_combined_avg=0.819.ckpt'

    qkv_outproj_base_1 = f'{base_path}/msr_dual_ensemble_small_large_qkv_outproj/1/epoch=03-val_rho_combined_avg=0.815.ckpt'
    qkv_outproj_base_2 = f'{base_path}/msr_dual_ensemble_small_large_qkv_outproj/2/epoch=03-val_rho_combined_avg=0.816.ckpt'
    qkv_outproj_base_3 = f'{base_path}/msr_dual_ensemble_small_large_qkv_outproj/3/epoch=05-val_rho_combined_avg=0.818.ckpt'

    noqkv_base_1 = f'{base_path}/msr_dual_ensemble_small_large_noqkv/1/epoch=01-val_rho_combined_avg=0.819.ckpt'
    noqkv_base_2 = f'{base_path}/msr_dual_ensemble_small_large_noqkv/2/epoch=02-val_rho_combined_avg=0.817.ckpt'
    noqkv_base_3 = f'{base_path}/msr_dual_ensemble_small_large_noqkv/3/epoch=02-val_rho_combined_avg=0.819.ckpt'

    qkv_only_base_1 = f'{base_path}/msr_dual_ensemble_small_large_qkv_only/1/epoch=04-val_rho_combined_avg=0.813.ckpt'
    qkv_only_base_2 = f'{base_path}/msr_dual_ensemble_small_large_qkv_only/2/epoch=02-val_rho_combined_avg=0.813.ckpt'
    qkv_only_base_3 = f'{base_path}/msr_dual_ensemble_small_large_qkv_only/3/epoch=04-val_rho_combined_avg=0.813.ckpt'

    medmed_base_1 = f'{base_path}/msr_dual_ensemble_med_med/1/epoch=04-val_rho_combined_avg=0.819.ckpt'
    medmed_base_2 = f'{base_path}/msr_dual_ensemble_med_med/2/epoch=02-val_rho_combined_avg=0.818.ckpt'
    medmed_base_3 = f'{base_path}/msr_dual_ensemble_med_med/3/epoch=02-val_rho_combined_avg=0.819.ckpt'

    detach_reg_base_1 = f'{base_path}/msr_dual_ensemble_small_large_detach_reg/1/epoch=02-val_rho_combined_avg=0.823.ckpt'
    detach_reg_base_2 = f'{base_path}/msr_dual_ensemble_small_large_detach_reg/2/epoch=03-val_rho_combined_avg=0.820.ckpt'
    detach_reg_base_3 = f'{base_path}/msr_dual_ensemble_small_large_detach_reg/3/epoch=04-val_rho_combined_avg=0.820.ckpt'

    r1_base_1 = f'{base_path}/msr_dual_ensemble_small_large_r1/1/epoch=02-val_rho_combined_avg=0.810.ckpt'
    r1_base_2 = f'{base_path}/msr_dual_ensemble_small_large_r1/2/epoch=02-val_rho_combined_avg=0.812.ckpt'
    r1_base_3 = f'{base_path}/msr_dual_ensemble_small_large_r1/3/epoch=03-val_rho_combined_avg=0.811.ckpt'

    fused_base_1 = f'{base_path}/msr_dual_ensemble_small_large_fused/1/epoch=01-val_rho_combined_avg=0.814.ckpt'
    fused_base_2 = f'{base_path}/msr_dual_ensemble_small_large_fused/2/epoch=02-val_rho_combined_avg=0.812.ckpt'
    fused_base_3 = f'{base_path}/msr_dual_ensemble_small_large_fused/3/epoch=03-val_rho_combined_avg=0.813.ckpt'

    fused_base = f'{base_path}/msr_dual_ensemble_fused_large/1/epoch=05-val_rho_combined_avg=0.809.ckpt'
    
    # ================= THERMOMPNN =================

    if dataset == 'domainome':

        # 1. Load Single Model Predictions (for single mutants)
        df_tm_1_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_1_epoch=99_val_ddG_spearman=0.73.ckpt.csv', index_col=0).rename(columns={'ddG_pred': 'ThermoMPNN_1'})
        df_tm_2_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_2_epoch=98_val_ddG_spearman=0.73.ckpt.csv', index_col=0).rename(columns={'ddG_pred': 'ThermoMPNN_2'})
        df_tm_3_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_3_epoch=99_val_ddG_spearman=0.73.ckpt.csv', index_col=0).rename(columns={'ddG_pred': 'ThermoMPNN_3'})

        # Assemble Thermo Ensemble
        df_thermo = df_tm_1_single.join(df_tm_2_single[['ThermoMPNN_2']], how='inner').join(df_tm_3_single[['ThermoMPNN_3']], how='inner').rename(columns={'ddG_true': 'ddG_ML_thermo'})
        # Attach ground truth from the first replicate
        df_thermo['ThermoMPNN(-D)_1'] = df_thermo['ThermoMPNN_1']
        df_thermo['ThermoMPNN(-D)_2'] = df_thermo['ThermoMPNN_2']
        df_thermo['ThermoMPNN(-D)_3'] = df_thermo['ThermoMPNN_3']

    else:

        # 1. Load Single Model Predictions (for single mutants)
        df_tm_1_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_1_epoch=99_val_ddG_spearman=0.73.ckpt_singles.csv', index_col=0)
        df_tm_2_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_2_epoch=98_val_ddG_spearman=0.73.ckpt_singles.csv', index_col=0)
        df_tm_3_single = pd.read_csv(f'{base_path}/thermompnnd/single_hyperopt_splits_correct_bb_3_epoch=99_val_ddG_spearman=0.73.ckpt_singles.csv', index_col=0)

        # 2. Load Epistatic Model Predictions (for multimutants)
        df_tm_1_epistatic = pd.read_csv(f'{base_path}/thermompnnd/epistatic_hyperopt_splits_correct_bb_1_epoch=84_val_ddG_spearman=0.74.ckpt.csv', index_col=0)
        df_tm_2_epistatic = pd.read_csv(f'{base_path}/thermompnnd/epistatic_hyperopt_splits_correct_bb_2_epoch=56_val_ddG_spearman=0.73.ckpt.csv', index_col=0)
        df_tm_3_epistatic = pd.read_csv(f'{base_path}/thermompnnd/epistatic_hyperopt_splits_correct_bb_3_epoch=95_val_ddG_spearman=0.73.ckpt.csv', index_col=0)

        # 3. Load Additive Model Predictions (for multimutant baseline)
        df_tm_1_additive = pd.read_csv(f'{base_path}/thermompnnd/additive_single_hyperopt_splits_correct_bb_1_epoch=99_val_ddG_spearman=0.73.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')
        df_tm_2_additive = pd.read_csv(f'{base_path}/thermompnnd/additive_single_hyperopt_splits_correct_bb_2_epoch=98_val_ddG_spearman=0.73.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')
        df_tm_3_additive = pd.read_csv(f'{base_path}/thermompnnd/additive_single_hyperopt_splits_correct_bb_3_epoch=99_val_ddG_spearman=0.73.ckpt.csv', index_col=0).dropna(subset='ddG_pred_additive')

        # --- Construct ThermoMPNN column (Singles + Additive Multimutants) ---
        df_tm_1_add_col = pd.concat([df_tm_1_single['ddG_pred'], df_tm_1_additive['ddG_pred_additive']]).to_frame('ThermoMPNN_1')
        df_tm_2_add_col = pd.concat([df_tm_2_single['ddG_pred'], df_tm_2_additive['ddG_pred_additive']]).to_frame('ThermoMPNN_2')
        df_tm_3_add_col = pd.concat([df_tm_3_single['ddG_pred'], df_tm_3_additive['ddG_pred_additive']]).to_frame('ThermoMPNN_3')

        # --- Construct ThermoMPNN(-D) column (Epistatic Multimutants filled with Singles) ---
        # FLAW: We must strictly filter the single model to only include non-colon indices 
        # to prevent "additive" leaking into the "epistatic" column per prompt instructions.
        df_tm_1_epi_col = df_tm_1_epistatic['ddG_pred'].rename('ThermoMPNN(-D)_1')
        df_tm_1_epi_col = df_tm_1_epi_col.combine_first(df_tm_1_single.loc[~df_tm_1_single.index.str.contains(':'), 'ddG_pred'].rename('ThermoMPNN(-D)_1'))

        df_tm_2_epi_col = df_tm_2_epistatic['ddG_pred'].rename('ThermoMPNN(-D)_2')
        df_tm_2_epi_col = df_tm_2_epi_col.combine_first(df_tm_2_single.loc[~df_tm_2_single.index.str.contains(':'), 'ddG_pred'].rename('ThermoMPNN(-D)_2'))

        df_tm_3_epi_col = df_tm_3_epistatic['ddG_pred'].rename('ThermoMPNN(-D)_3')
        df_tm_3_epi_col = df_tm_3_epi_col.combine_first(df_tm_3_single.loc[~df_tm_3_single.index.str.contains(':'), 'ddG_pred'].rename('ThermoMPNN(-D)_3'))

        # Assemble Thermo Ensemble
        df_thermo = df_tm_1_add_col.join([df_tm_1_epi_col, df_tm_2_add_col, df_tm_2_epi_col, df_tm_3_add_col, df_tm_3_epi_col], how='inner')
        # Attach ground truth from the first replicate
        df_thermo = df_thermo.join(df_tm_1_single[['code', 'mut_type', 'ddG_true']].rename(columns={'ddG_true': 'ddG_ML_thermo'}))
        df_thermo['ThermoMPNN(-D)_additive_1'] = df_thermo['ThermoMPNN_1']
        df_thermo['ThermoMPNN(-D)_additive_2'] = df_thermo['ThermoMPNN_2']
        df_thermo['ThermoMPNN(-D)_additive_3'] = df_thermo['ThermoMPNN_3']

    # ================= SPURS =================

    df_spurs_1 = pd.read_csv(f'{base_path}/SPURS/step_0-val_rho_avg_spearman_0.69_single_seed1.ckpt+step_0-val_rho_avg_spearman_0.77_multi_seed1.ckpt.csv', index_col=0).rename(columns={'SPURS_single_additive': 'SPURS_additive_1', 'SPURS_multi': 'SPURS_1'})
    df_spurs_2 = pd.read_csv(f'{base_path}/SPURS/step_0-val_rho_avg_spearman_0.68_single_seed2.ckpt+step_0-val_rho_avg_spearman_0.77_multi_seed2.ckpt.csv', index_col=0).rename(columns={'SPURS_single_additive': 'SPURS_additive_2', 'SPURS_multi': 'SPURS_2'})
    df_spurs_3 = pd.read_csv(f'{base_path}/SPURS/step_0-val_rho_avg_spearman_0.69_single_seed3.ckpt+step_0-val_rho_avg_spearman_0.75_multi_seed3.ckpt.csv', index_col=0).rename(columns={'SPURS_single_additive': 'SPURS_additive_3', 'SPURS_multi': 'SPURS_3'})

    df_spurs = pd.concat([df_spurs_1, df_spurs_2[[c for c in df_spurs_2.columns if 'SPURS' in c]], df_spurs_3[[c for c in df_spurs_3.columns if 'SPURS' in c]]], axis=1)
    df_spurs.index.name = 'uid'

    df_spurs_reported = pd.read_csv(f'{base_path}/SPURS/hf+hf.csv', index_col=0).rename(columns={'SPURS_single_additive': 'SPURS_reported_additive', 'SPURS_multi': 'SPURS_reported'})

    df_spurs_reported.index.name = 'uid'

    # ================= MUTATE EVERYTHING =================

    df_me_1 = pd.read_csv(f'{base_path}/mutate_everything/af_double_hyperopt_splits_1_checkpoint-99.csv', index_col=0)
    df_me_1['MutateEverything_1'] = df_me_1['ddG_pred']
    df_me_1['ddG_ML_me'] = df_me_1['ddG_true']
    df_me_2 = pd.read_csv(f'{base_path}/mutate_everything/af_double_hyperopt_splits_2_checkpoint-99.csv', index_col=0)
    df_me_2['MutateEverything_2'] = df_me_2['ddG_pred']
    df_me_3 = pd.read_csv(f'{base_path}/mutate_everything/af_double_hyperopt_splits_3_checkpoint-99.csv', index_col=0)
    df_me_3['MutateEverything_3'] = df_me_3['ddG_pred']

    df_me = df_me_1[['code', 'mut_type', 'MutateEverything_1', 'ddG_ML_me']].join(df_me_2[['MutateEverything_2']], how='inner').join(df_me_3[['MutateEverything_3']], how='inner')

    assert len(df_me) == len(df_me_1)

    df_me_single_1 = pd.read_csv(f'{base_path}/mutate_everything/af_single_hyperopt_splits_1_checkpoint-19.csv', index_col=0)
    df_me_single_1['MutateEverything_single_1'] = df_me_single_1['ddG_pred']
    df_me_single_1['ddG_ML_me'] = df_me_single_1['ddG_true']
    df_me_single_2 = pd.read_csv(f'{base_path}/mutate_everything/af_single_hyperopt_splits_2_checkpoint-19.csv', index_col=0)
    df_me_single_2['MutateEverything_single_2'] = df_me_single_2['ddG_pred']
    df_me_single_3 = pd.read_csv(f'{base_path}/mutate_everything/af_single_hyperopt_splits_3_checkpoint-19.csv', index_col=0)
    df_me_single_3['MutateEverything_single_3'] = df_me_single_3['ddG_pred']

    df_me_single = df_me_single_1[['code', 'mut_type', 'MutateEverything_single_1', 'ddG_ML_me']].join(df_me_single_2[['MutateEverything_single_2']], how='inner').join(df_me_single_3[['MutateEverything_single_3']], how='inner')

    df_me_additive_1 = pd.read_csv(f'{base_path}/mutate_everything/af_single_additive_hyperopt_splits_1_checkpoint-19.csv', index_col=0)
    df_me_additive_1 = add_additive_predictions(df_me_additive_1, df_me)
    df_me_additive_1 = df_me_additive_1[['code', 'mut_type', 'ddG_pred_additive']].rename(columns={'ddG_pred_additive': 'MutateEverything_additive_1'})

    df_me_additive_2 = pd.read_csv(f'{base_path}/mutate_everything/af_single_additive_hyperopt_splits_2_checkpoint-19.csv', index_col=0)
    df_me_additive_2 = add_additive_predictions(df_me_additive_2, df_me)[['code', 'mut_type', 'ddG_pred_additive']].rename(columns={'ddG_pred_additive': 'MutateEverything_additive_2'})

    df_me_additive_3 = pd.read_csv(f'{base_path}/mutate_everything/af_single_additive_hyperopt_splits_3_checkpoint-19.csv', index_col=0)
    df_me_additive_3 = add_additive_predictions(df_me_additive_3, df_me)[['code', 'mut_type', 'ddG_pred_additive']].rename(columns={'ddG_pred_additive': 'MutateEverything_additive_3'})

    df_me_additive = df_me_additive_1[['MutateEverything_additive_1']].join(df_me_additive_2[['MutateEverything_additive_2']], how='inner').join(df_me_additive_3[['MutateEverything_additive_3']], how='inner')
    assert len(df_me_additive) == len(df_me)
    df_me = df_me.join(df_me_additive)
    df_me['uid'] = df_me['code'] + '_' + df_me['mut_type']
    df_me = df_me.set_index('uid')

    # ================= ZERO SHOT & BASELINES =================

    df_zero_sm = pd.read_csv(f'{base_path}/esm3_forge/zero_shot/esm3-small-2024-08/predictions_unmasked.csv', index_col=0).rename(columns={'esm3_score': 'ESM3-small', 'pred_additive': 'ESM3-small_additive'})
    df_zero_sm = df_zero_sm.reset_index(drop=True)
    df_zero_sm['uid'] = df_zero_sm['code'] + '_' + df_zero_sm['mut_type']
    df_zero_sm = df_zero_sm.set_index('uid')
    df_zero_med = pd.read_csv(f'{base_path}/esm3_forge/zero_shot/esm3-medium-2024-08/predictions_unmasked.csv', index_col=0).rename(columns={'esm3_score': 'ESM3-medium', 'pred_additive': 'ESM3-medium_additive'})
    df_zero_med = df_zero_med.reset_index(drop=True)
    df_zero_med['uid'] = df_zero_med['code'] + '_' + df_zero_med['mut_type']
    df_zero_med = df_zero_med.set_index('uid')
    df_zero_lg = pd.read_csv(f'{base_path}/esm3_forge/zero_shot/esm3-large-2024-03/predictions_unmasked.csv', index_col=0).rename(columns={'esm3_score': 'ESM3-large', 'pred_additive': 'ESM3-large_additive'})
    df_zero_lg = df_zero_lg.reset_index(drop=True)
    df_zero_lg['uid'] = df_zero_lg['code'] + '_' + df_zero_lg['mut_type']
    df_zero_lg = df_zero_lg.set_index('uid')

    try:
        df_mpnn = pd.read_csv(f'{base_path}/proteinmpnn/proteinmpnn_020_unmasked.csv', index_col=0).reset_index(drop=True).rename(columns={'mpnn_score': 'ProteinMPNN', 'mpnn_score_additive': 'ProteinMPNN_additive', 'uid.1': 'uid'}).set_index('uid')
    except:
        df_mpnn = pd.read_csv(f'{base_path}/proteinmpnn/proteinmpnn_020_unmasked.csv', index_col=0).rename(columns={'mpnn_score': 'ProteinMPNN', 'mpnn_score_additive': 'ProteinMPNN_additive'})
    
    try:
        df_mpnn_masked = pd.read_csv(f'{base_path}/proteinmpnn/proteinmpnn_020.csv', index_col=0).reset_index(drop=True).rename(columns={'mpnn_score': 'ProteinMPNN_masked', 'mpnn_score_additive': 'ProteinMPNN_masked_additive', 'uid.1': 'uid'}).set_index('uid')
    except:
        df_mpnn_masked = pd.read_csv(f'{base_path}/proteinmpnn/proteinmpnn_020.csv', index_col=0).rename(columns={'mpnn_score': 'ProteinMPNN_masked', 'mpnn_score_additive': 'ProteinMPNN_masked_additive'})

    try:
        # Rosetta Cartesian DDG
        df_ros = pd.read_csv(f'/home/sareeves/software/esm-msr/analysis_notebooks/predictions/{dataset}/rosetta/cartesian_ddg.csv', index_col=0)
        df_ros['code'] = df_ros.index.to_series().apply(lambda x: x.split('_')[0])
        df_ros['mut_type'] = df_ros.index.to_series().apply(lambda x: '_'.join(x.split('_')[1:]))
        df_ros['Rosetta Cartesian DDG_1'] = df_ros['cartesian_ddg_1'] / 2.94
        df_ros['ddG_pred'] = df_ros['cartesian_ddg_1'] / 2.94
        df_ros = add_additive_predictions(df_ros, df_ros)
        df_ros = df_ros.rename(columns={'ddG_pred_additive': 'Rosetta Cartesian DDG_additive_1'})
        df_ros['Rosetta Cartesian DDG_2'] = df_ros['cartesian_ddg_2'] / 2.94
        df_ros['ddG_pred'] = df_ros['cartesian_ddg_2'] / 2.94
        df_ros = add_additive_predictions(df_ros, df_ros)
        df_ros = df_ros.rename(columns={'ddG_pred_additive': 'Rosetta Cartesian DDG_additive_2'})
        df_ros['Rosetta Cartesian DDG_3'] = df_ros['cartesian_ddg_3'] / 2.94
        df_ros['ddG_pred'] = df_ros['cartesian_ddg_3'] / 2.94
        df_ros = add_additive_predictions(df_ros, df_ros)
        df_ros = df_ros.rename(columns={'ddG_pred_additive': 'Rosetta Cartesian DDG_additive_3'})

        df_ros.index.name = 'uid'
        df_ros = df_ros.reset_index()
        df_ros['code'] = df_ros['uid'].apply(lambda x: x.split('_')[0])
        df_ros['mut_type'] = df_ros['uid'].apply(lambda x: x.split('_')[-1])
        df_ros = df_ros.set_index('uid')
    except FileNotFoundError:
        df_ros = pd.DataFrame()

    # ================= ESM-MSR =================

    truth = 'ddG_ML'

    df_esm_1 = pd.read_csv(f'{msr_base_1}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_1['ESM-MSR_1'] = df_esm_1['combined_pred']
    try:
        df_esm_1['ESM-MSR_additive_1'] = df_esm_1['combined_pred_additive']
    except:
        pass
    df_esm_1['ddG_ML_esm'] = df_esm_1[truth]

    df_esm_2 = pd.read_csv(f'{msr_base_2}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_2['ESM-MSR_2'] = df_esm_2['combined_pred']
    try:
        df_esm_2['ESM-MSR_additive_2'] = df_esm_2['combined_pred_additive']
    except:
        pass

    df_esm_3 = pd.read_csv(f'{msr_base_3}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_3['ESM-MSR_3'] = df_esm_3['combined_pred']
    try:
        df_esm_3['ESM-MSR_additive_3'] = df_esm_3['combined_pred_additive']
    except:
        pass

    df_esm = pd.concat([df_esm_1, df_esm_2[[c for c in df_esm_2.columns if 'MSR' in c]], df_esm_3[[c for c in df_esm_3.columns if 'MSR' in c]]], axis=1)

    df_esm_wt_1 = pd.read_csv(f'{msr_base_1}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_wt_1['ESM-MSR_WT_1'] = df_esm_wt_1['wt_lora_pred']
    try:
        df_esm_wt_1['ESM-MSR_WT_additive_1'] = df_esm_wt_1['wt_lora_pred_additive']
    except:
        pass
    df_esm_wt_1['ddG_ML_esm'] = df_esm_wt_1[truth]

    df_esm_wt_2 = pd.read_csv(f'{msr_base_2}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_wt_2['ESM-MSR_WT_2'] = df_esm_wt_2['wt_lora_pred']
    try:
        df_esm_wt_2['ESM-MSR_WT_additive_2'] = df_esm_wt_2['wt_lora_pred_additive']
    except:
        pass

    df_esm_wt_3 = pd.read_csv(f'{msr_base_3}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_wt_3['ESM-MSR_WT_3'] = df_esm_wt_3['wt_lora_pred']
    try:
        df_esm_wt_3['ESM-MSR_WT_additive_3'] = df_esm_wt_3['wt_lora_pred_additive']
    except:
        pass

    df_esm_wt = pd.concat([df_esm_wt_1, df_esm_wt_2[[c for c in df_esm_wt_2.columns if 'MSR' in c]], df_esm_wt_3[[c for c in df_esm_wt_3.columns if 'MSR' in c]]], axis=1)

    df_esm_mt_1 = pd.read_csv(f'{msr_base_1}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_mt_1['ESM-MSR_MT_1'] = df_esm_mt_1['mt_lora_pred']
    try:
        df_esm_mt_1['ESM-MSR_MT_additive_1'] = df_esm_mt_1['mt_lora_pred_additive']
    except:
        pass
    df_esm_mt_1['ddG_ML_esm'] = df_esm_mt_1[truth]

    df_esm_mt_2 = pd.read_csv(f'{msr_base_2}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_mt_2['ESM-MSR_MT_2'] = df_esm_mt_2['mt_lora_pred']
    try:
        df_esm_mt_2['ESM-MSR_MT_additive_2'] = df_esm_mt_2['mt_lora_pred_additive']
    except:
        pass

    df_esm_mt_3 = pd.read_csv(f'{msr_base_3}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_mt_3['ESM-MSR_MT_3'] = df_esm_mt_3['mt_lora_pred']
    try:
        df_esm_mt_3['ESM-MSR_MT_additive_3'] = df_esm_mt_3['mt_lora_pred_additive']
    except:
        pass

    df_esm_mt = pd.concat([df_esm_mt_1, df_esm_mt_2[[c for c in df_esm_mt_2.columns if 'MSR' in c]], df_esm_mt_3[[c for c in df_esm_mt_3.columns if 'MSR' in c]]], axis=1)

    df_esm_defchain_1 = pd.read_csv(f'{msr_base_1}_alpha{full_alpha}_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_defchain_1['ESM-MSR_defchain_1'] = df_esm_defchain_1['combined_pred']
    try:
        df_esm_defchain_1['ESM-MSR_defchain_additive_1'] = df_esm_defchain_1['combined_pred_additive']
    except:
        pass
    df_esm_defchain_1['ddG_ML_esm'] = df_esm_defchain_1[truth]

    df_esm_defchain_2 = pd.read_csv(f'{msr_base_2}_alpha{full_alpha}_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_defchain_2['ESM-MSR_defchain_2'] = df_esm_defchain_2['combined_pred']
    try:
        df_esm_defchain_2['ESM-MSR_defchain_additive_2'] = df_esm_defchain_2['combined_pred_additive']
    except:
        pass

    df_esm_defchain_3 = pd.read_csv(f'{msr_base_3}_alpha{full_alpha}_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_defchain_3['ESM-MSR_defchain_3'] = df_esm_defchain_3['combined_pred']
    try:
        df_esm_defchain_3['ESM-MSR_defchain_additive_3'] = df_esm_defchain_3['combined_pred_additive']
    except:
        pass

    df_esm_defchain = pd.concat([df_esm_defchain_1, df_esm_defchain_2[[c for c in df_esm_defchain_2.columns if 'MSR' in c]], df_esm_defchain_3[[c for c in df_esm_defchain_3.columns if 'MSR' in c]]], axis=1)

    try:
        df_esm_defmarginal_1 = pd.read_csv(f'{msr_base_1}_alpha{full_alpha}_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_defmarginal_1['ESM-MSR_defmarginal_1'] = df_esm_defmarginal_1['combined_pred']
        try:
            df_esm_defmarginal_1['ESM-MSR_defmarginal_additive_1'] = df_esm_defmarginal_1['combined_pred_additive']
        except:
            pass
        df_esm_defmarginal_1['ddG_ML_esm'] = df_esm_defmarginal_1[truth]

        df_esm_defmarginal_2 = pd.read_csv(f'{msr_base_2}_alpha{full_alpha}_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_defmarginal_2['ESM-MSR_defmarginal_2'] = df_esm_defmarginal_2['combined_pred']
        try:
            df_esm_defmarginal_2['ESM-MSR_defmarginal_additive_2'] = df_esm_defmarginal_2['combined_pred_additive']
        except:
            pass

        df_esm_defmarginal_3 = pd.read_csv(f'{msr_base_3}_alpha{full_alpha}_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_defmarginal_3['ESM-MSR_defmarginal_3'] = df_esm_defmarginal_3['combined_pred']
        try:
            df_esm_defmarginal_3['ESM-MSR_defmarginal_additive_3'] = df_esm_defmarginal_3['combined_pred_additive']
        except:
            pass

        df_esm_defmarginal = pd.concat([df_esm_defmarginal_1, df_esm_defmarginal_2[[c for c in df_esm_defmarginal_2.columns if 'MSR' in c]], df_esm_defmarginal_3[[c for c in df_esm_defmarginal_3.columns if 'MSR' in c]]], axis=1)
    except Exception as e:
        print(e)
        df_esm_defmarginal = df_esm_defchain

    df_esm_alpha_high_1 = pd.read_csv(f'{msr_base_1}_alpha{high_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_high_1['ESM-MSR_alpha_high_1'] = df_esm_alpha_high_1['combined_pred']
    try:
        df_esm_alpha_high_1['ESM-MSR_alpha_high_additive_1'] = df_esm_alpha_high_1['combined_pred_additive']
    except:
        pass
    df_esm_alpha_high_1['ddG_ML_esm'] = df_esm_alpha_high_1[truth]

    df_esm_alpha_high_2 = pd.read_csv(f'{msr_base_2}_alpha{high_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_high_2['ESM-MSR_alpha_high_2'] = df_esm_alpha_high_2['combined_pred']
    try:
        df_esm_alpha_high_2['ESM-MSR_alpha_high_additive_2'] = df_esm_alpha_high_2['combined_pred_additive']
    except:
        pass

    df_esm_alpha_high_3 = pd.read_csv(f'{msr_base_3}_alpha{high_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_high_3['ESM-MSR_alpha_high_3'] = df_esm_alpha_high_3['combined_pred']
    try:
        df_esm_alpha_high_3['ESM-MSR_alpha_high_additive_3'] = df_esm_alpha_high_3['combined_pred_additive']
    except:
        pass

    df_esm_alpha_high = pd.concat([df_esm_alpha_high_1, df_esm_alpha_high_2[[c for c in df_esm_alpha_high_2.columns if 'MSR' in c]], df_esm_alpha_high_3[[c for c in df_esm_alpha_high_3.columns if 'MSR' in c]]], axis=1)

    df_esm_alpha_med_1 = pd.read_csv(f'{msr_base_1}_alpha{med_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_med_1['ESM-MSR_alpha_med_1'] = df_esm_alpha_med_1['combined_pred']
    try:
        df_esm_alpha_med_1['ESM-MSR_alpha_med_additive_1'] = df_esm_alpha_med_1['combined_pred_additive']
    except:
        pass
    df_esm_alpha_med_1['ddG_ML_esm'] = df_esm_alpha_med_1[truth]

    df_esm_alpha_med_2 = pd.read_csv(f'{msr_base_2}_alpha{med_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_med_2['ESM-MSR_alpha_med_2'] = df_esm_alpha_med_2['combined_pred']
    try:
        df_esm_alpha_med_2['ESM-MSR_alpha_med_additive_2'] = df_esm_alpha_med_2['combined_pred_additive']
    except:
        pass

    df_esm_alpha_med_3 = pd.read_csv(f'{msr_base_3}_alpha{med_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_med_3['ESM-MSR_alpha_med_3'] = df_esm_alpha_med_3['combined_pred']
    try:
        df_esm_alpha_med_3['ESM-MSR_alpha_med_additive_3'] = df_esm_alpha_med_3['combined_pred_additive']
    except:
        pass

    df_esm_alpha_med = pd.concat([df_esm_alpha_med_1, df_esm_alpha_med_2[[c for c in df_esm_alpha_med_2.columns if 'MSR' in c]], df_esm_alpha_med_3[[c for c in df_esm_alpha_med_3.columns if 'MSR' in c]]], axis=1)

    df_esm_alpha_low_1 = pd.read_csv(f'{msr_base_1}_alpha{low_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_low_1['ESM-MSR_alpha_low_1'] = df_esm_alpha_low_1['combined_pred']
    try:
        df_esm_alpha_low_1['ESM-MSR_alpha_low_additive_1'] = df_esm_alpha_low_1['combined_pred_additive']
    except:
        pass
    df_esm_alpha_low_1['ddG_ML_esm'] = df_esm_alpha_low_1[truth]

    df_esm_alpha_low_2 = pd.read_csv(f'{msr_base_2}_alpha{low_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_low_2['ESM-MSR_alpha_low_2'] = df_esm_alpha_low_2['combined_pred']
    try:
        df_esm_alpha_low_2['ESM-MSR_alpha_low_additive_2'] = df_esm_alpha_low_2['combined_pred_additive']
    except:
        pass

    df_esm_alpha_low_3 = pd.read_csv(f'{msr_base_3}_alpha{low_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_alpha_low_3['ESM-MSR_alpha_low_3'] = df_esm_alpha_low_3['combined_pred']
    try:
        df_esm_alpha_low_3['ESM-MSR_alpha_low_additive_3'] = df_esm_alpha_low_3['combined_pred_additive']
    except:
        pass

    df_esm_alpha_low = pd.concat([df_esm_alpha_low_1, df_esm_alpha_low_2[[c for c in df_esm_alpha_low_2.columns if 'MSR' in c]], df_esm_alpha_low_3[[c for c in df_esm_alpha_low_3.columns if 'MSR' in c]]], axis=1)

    df_esm_single_1 = pd.read_csv(f'{single_base_1}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_single_1['ESM-MSR_single_1'] = df_esm_single_1['combined_pred']
    try:
        df_esm_single_1['ESM-MSR_single_additive_1'] = df_esm_single_1['combined_pred_additive']
    except:
        pass
    df_esm_single_1['ddG_ML_esm'] = df_esm_single_1[truth]

    df_esm_single_2 = pd.read_csv(f'{single_base_2}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_single_2['ESM-MSR_single_2'] = df_esm_single_2['combined_pred']
    try:
        df_esm_single_2['ESM-MSR_single_additive_2'] = df_esm_single_2['combined_pred_additive']
    except:
        pass

    df_esm_single_3 = pd.read_csv(f'{single_base_3}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_single_3['ESM-MSR_single_3'] = df_esm_single_3['combined_pred']
    try:
        df_esm_single_3['ESM-MSR_single_additive_3'] = df_esm_single_3['combined_pred_additive']
    except:
        pass

    df_esm_single = pd.concat([df_esm_single_1, df_esm_single_2[[c for c in df_esm_single_2.columns if 'MSR' in c]], df_esm_single_3[[c for c in df_esm_single_3.columns if 'MSR' in c]]], axis=1)

    try:
        df_esm_reg_1 = pd.read_csv(f'{reg_base_1}_alpha{full_alpha}_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_1['ESM-MSR_reg_1'] = df_esm_reg_1['combined_pred']
        try:
            df_esm_reg_1['ESM-MSR_reg_additive_1'] = df_esm_reg_1['combined_pred_additive']
        except:
            pass
        df_esm_reg_1['ddG_ML_esm'] = df_esm_reg_1[truth]

        df_esm_reg_2 = pd.read_csv(f'{reg_base_2}_alpha{full_alpha}_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_2['ESM-MSR_reg_2'] = df_esm_reg_2['combined_pred']
        try:
            df_esm_reg_2['ESM-MSR_reg_additive_2'] = df_esm_reg_2['combined_pred_additive']
        except:
            pass

        df_esm_reg_3 = pd.read_csv(f'{reg_base_3}_alpha{full_alpha}_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_3['ESM-MSR_reg_3'] = df_esm_reg_3['combined_pred']
        try:
            df_esm_reg_3['ESM-MSR_reg_additive_3'] = df_esm_reg_3['combined_pred_additive']
        except:
            pass

        df_esm_reg = pd.concat([df_esm_reg_1, df_esm_reg_2[[c for c in df_esm_reg_2.columns if 'MSR' in c]], df_esm_reg_3[[c for c in df_esm_reg_3.columns if 'MSR' in c]]], axis=1)

    except:
        df_esm_reg_1 = pd.read_csv(f'{reg_base_1}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_1['ESM-MSR_reg_1'] = df_esm_reg_1['combined_pred']
        try:
            df_esm_reg_1['ESM-MSR_reg_additive_1'] = df_esm_reg_1['combined_pred_additive']
        except:
            pass
        df_esm_reg_1['ddG_ML_esm'] = df_esm_reg_1[truth]

        df_esm_reg_2 = pd.read_csv(f'{reg_base_2}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_2['ESM-MSR_reg_2'] = df_esm_reg_2['combined_pred']
        try:
            df_esm_reg_2['ESM-MSR_reg_additive_2'] = df_esm_reg_2['combined_pred_additive']
        except:
            pass

        df_esm_reg_3 = pd.read_csv(f'{reg_base_3}_alpha{full_alpha}_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
        df_esm_reg_3['ESM-MSR_reg_3'] = df_esm_reg_3['combined_pred']
        try:
            df_esm_reg_3['ESM-MSR_reg_additive_3'] = df_esm_reg_3['combined_pred_additive']
        except:
            pass

        df_esm_reg = pd.concat([df_esm_reg_1, df_esm_reg_2[[c for c in df_esm_reg_2.columns if 'MSR' in c]], df_esm_reg_3[[c for c in df_esm_reg_3.columns if 'MSR' in c]]], axis=1)

    df_esm_masked_1 = pd.read_csv(f'{mm_base_1}_alpha{int(float(full_alpha))}.0_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_masked_1['ESM-MSR_masked_1'] = df_esm_masked_1['combined_pred']
    try:
        df_esm_masked_1['ESM-MSR_masked_additive_1'] = df_esm_masked_1['combined_pred_additive']
    except:
        pass
    df_esm_masked_1['ddG_ML_esm'] = df_esm_masked_1[truth]

    df_esm_masked_2 = pd.read_csv(f'{mm_base_2}_alpha{int(float(full_alpha))}.0_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_masked_2['ESM-MSR_masked_2'] = df_esm_masked_2['combined_pred']
    try:
        df_esm_masked_2['ESM-MSR_masked_additive_2'] = df_esm_masked_2['combined_pred_additive']
    except:
        pass

    df_esm_masked_3 = pd.read_csv(f'{mm_base_3}_alpha{int(float(full_alpha))}.0_marginal_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_masked_3['ESM-MSR_masked_3'] = df_esm_masked_3['combined_pred']
    try:
        df_esm_masked_3['ESM-MSR_masked_additive_3'] = df_esm_masked_3['combined_pred_additive']
    except:
        pass

    df_esm_masked = pd.concat([df_esm_masked_1, df_esm_masked_2[[c for c in df_esm_masked_2.columns if 'MSR' in c]], df_esm_masked_3[[c for c in df_esm_masked_3.columns if 'MSR' in c]]], axis=1)


    df_esm_chain_1 = pd.read_csv(f'{chain_base_1}_alpha{int(float(full_alpha))}.0_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_chain_1['ESM-MSR_chain_1'] = df_esm_chain_1['combined_pred']
    try:
        df_esm_chain_1['ESM-MSR_chain_additive_1'] = df_esm_chain_1['combined_pred_additive']
    except:
        pass
    df_esm_chain_1['ddG_ML_esm'] = df_esm_chain_1[truth]

    df_esm_chain_2 = pd.read_csv(f'{chain_base_2}_alpha{int(float(full_alpha))}.0_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_chain_2['ESM-MSR_chain_2'] = df_esm_chain_2['combined_pred']
    try:
        df_esm_chain_2['ESM-MSR_chain_additive_2'] = df_esm_chain_2['combined_pred_additive']
    except:
        pass

    df_esm_chain_3 = pd.read_csv(f'{chain_base_3}_alpha{int(float(full_alpha))}.0_chain_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_chain_3['ESM-MSR_chain_3'] = df_esm_chain_3['combined_pred']
    try:
        df_esm_chain_3['ESM-MSR_chain_additive_3'] = df_esm_chain_3['combined_pred_additive']
    except:
        pass

    df_esm_chain = pd.concat([df_esm_chain_1, df_esm_chain_2[[c for c in df_esm_chain_2.columns if 'MSR' in c]], df_esm_chain_3[[c for c in df_esm_chain_3.columns if 'MSR' in c]]], axis=1)


    df_esm_noseqhead_1 = pd.read_csv(f'{noseqhead_base_1}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noseqhead_1['ESM-MSR_noseqhead_1'] = df_esm_noseqhead_1['combined_pred']
    try:
        df_esm_noseqhead_1['ESM-MSR_noseqhead_additive_1'] = df_esm_noseqhead_1['combined_pred_additive']
    except:
        pass
    df_esm_noseqhead_1['ddG_ML_esm'] = df_esm_noseqhead_1[truth]

    df_esm_noseqhead_2 = pd.read_csv(f'{noseqhead_base_2}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noseqhead_2['ESM-MSR_noseqhead_2'] = df_esm_noseqhead_2['combined_pred']
    try:
        df_esm_noseqhead_2['ESM-MSR_noseqhead_additive_2'] = df_esm_noseqhead_2['combined_pred_additive']
    except:
        pass

    df_esm_noseqhead_3 = pd.read_csv(f'{noseqhead_base_3}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noseqhead_3['ESM-MSR_noseqhead_3'] = df_esm_noseqhead_3['combined_pred']
    try:
        df_esm_noseqhead_3['ESM-MSR_noseqhead_additive_3'] = df_esm_noseqhead_3['combined_pred_additive']
    except:
        pass

    df_esm_noseqhead = pd.concat([df_esm_noseqhead_1, df_esm_noseqhead_2[[c for c in df_esm_noseqhead_2.columns if 'MSR' in c]], df_esm_noseqhead_3[[c for c in df_esm_noseqhead_3.columns if 'MSR' in c]]], axis=1)

    df_esm_all_1 = pd.read_csv(f'{all_base_1}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_all_1['ESM-MSR_all_1'] = df_esm_all_1['combined_pred']
    try:
        df_esm_all_1['ESM-MSR_all_additive_1'] = df_esm_all_1['combined_pred_additive']
    except:
        pass
    df_esm_all_1['ddG_ML_esm'] = df_esm_all_1[truth]

    df_esm_all_2 = pd.read_csv(f'{all_base_2}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_all_2['ESM-MSR_all_2'] = df_esm_all_2['combined_pred']
    try:
        df_esm_all_2['ESM-MSR_all_additive_2'] = df_esm_all_2['combined_pred_additive']
    except:
        pass

    df_esm_all_3 = pd.read_csv(f'{all_base_3}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_all_3['ESM-MSR_all_3'] = df_esm_all_3['combined_pred']
    try:
        df_esm_all_3['ESM-MSR_all_additive_3'] = df_esm_all_3['combined_pred_additive']
    except:
        pass

    df_esm_all = pd.concat([df_esm_all_1, df_esm_all_2[[c for c in df_esm_all_2.columns if 'MSR' in c]], df_esm_all_3[[c for c in df_esm_all_3.columns if 'MSR' in c]]], axis=1)


    df_esm_ffn_1 = pd.read_csv(f'{ffn_base_1}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_ffn_1['ESM-MSR_ffn_1'] = df_esm_ffn_1['combined_pred']
    try:
        df_esm_ffn_1['ESM-MSR_ffn_additive_1'] = df_esm_ffn_1['combined_pred_additive']
    except:
        pass
    df_esm_ffn_1['ddG_ML_esm'] = df_esm_ffn_1[truth]

    df_esm_ffn_2 = pd.read_csv(f'{ffn_base_2}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_ffn_2['ESM-MSR_ffn_2'] = df_esm_ffn_2['combined_pred']
    try:
        df_esm_ffn_2['ESM-MSR_ffn_additive_2'] = df_esm_ffn_2['combined_pred_additive']
    except:
        pass

    df_esm_ffn_3 = pd.read_csv(f'{ffn_base_3}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_ffn_3['ESM-MSR_ffn_3'] = df_esm_ffn_3['combined_pred']
    try:
        df_esm_ffn_3['ESM-MSR_ffn_additive_3'] = df_esm_ffn_3['combined_pred_additive']
    except:
        pass

    df_esm_ffn = pd.concat([df_esm_ffn_1, df_esm_ffn_2[[c for c in df_esm_ffn_2.columns if 'MSR' in c]], df_esm_ffn_3[[c for c in df_esm_ffn_3.columns if 'MSR' in c]]], axis=1)

    df_esm_qkv_outproj_1 = pd.read_csv(f'{qkv_outproj_base_1}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_outproj_1['ESM-MSR_qkv_outproj_1'] = df_esm_qkv_outproj_1['combined_pred']
    try:
        df_esm_qkv_outproj_1['ESM-MSR_qkv_outproj_additive_1'] = df_esm_qkv_outproj_1['combined_pred_additive']
    except:
        pass
    df_esm_qkv_outproj_1['ddG_ML_esm'] = df_esm_qkv_outproj_1[truth]

    df_esm_qkv_outproj_2 = pd.read_csv(f'{qkv_outproj_base_2}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_outproj_2['ESM-MSR_qkv_outproj_2'] = df_esm_qkv_outproj_2['combined_pred']
    try:
        df_esm_qkv_outproj_2['ESM-MSR_qkv_outproj_additive_2'] = df_esm_qkv_outproj_2['combined_pred_additive']
    except:
        pass

    df_esm_qkv_outproj_3 = pd.read_csv(f'{qkv_outproj_base_3}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_outproj_3['ESM-MSR_qkv_outproj_3'] = df_esm_qkv_outproj_3['combined_pred']
    try:
        df_esm_qkv_outproj_3['ESM-MSR_qkv_outproj_additive_3'] = df_esm_qkv_outproj_3['combined_pred_additive']
    except:
        pass

    df_esm_qkv_outproj = pd.concat([df_esm_qkv_outproj_1, df_esm_qkv_outproj_2[[c for c in df_esm_qkv_outproj_2.columns if 'MSR' in c]], df_esm_qkv_outproj_3[[c for c in df_esm_qkv_outproj_3.columns if 'MSR' in c]]], axis=1)

    df_esm_noqkv_1 = pd.read_csv(f'{noqkv_base_1}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noqkv_1['ESM-MSR_noqkv_1'] = df_esm_noqkv_1['combined_pred']
    try:
        df_esm_noqkv_1['ESM-MSR_noqkv_additive_1'] = df_esm_noqkv_1['combined_pred_additive']
    except:
        pass
    df_esm_noqkv_1['ddG_ML_esm'] = df_esm_noqkv_1[truth]

    df_esm_noqkv_2 = pd.read_csv(f'{noqkv_base_2}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noqkv_2['ESM-MSR_noqkv_2'] = df_esm_noqkv_2['combined_pred']
    try:
        df_esm_noqkv_2['ESM-MSR_noqkv_additive_2'] = df_esm_noqkv_2['combined_pred_additive']
    except:
        pass

    df_esm_noqkv_3 = pd.read_csv(f'{noqkv_base_3}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_noqkv_3['ESM-MSR_noqkv_3'] = df_esm_noqkv_3['combined_pred']
    try:
        df_esm_noqkv_3['ESM-MSR_noqkv_additive_3'] = df_esm_noqkv_3['combined_pred_additive']
    except:
        pass

    df_esm_noqkv = pd.concat([df_esm_noqkv_1, df_esm_noqkv_2[[c for c in df_esm_noqkv_2.columns if 'MSR' in c]], df_esm_noqkv_3[[c for c in df_esm_noqkv_3.columns if 'MSR' in c]]], axis=1)

    df_esm_qkv_only_1 = pd.read_csv(f'{qkv_only_base_1}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_only_1['ESM-MSR_qkv_only_1'] = df_esm_qkv_only_1['combined_pred']
    try:
        df_esm_qkv_only_1['ESM-MSR_qkv_only_additive_1'] = df_esm_qkv_only_1['combined_pred_additive']
    except:
        pass
    df_esm_qkv_only_1['ddG_ML_esm'] = df_esm_qkv_only_1[truth]

    df_esm_qkv_only_2 = pd.read_csv(f'{qkv_only_base_2}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_only_2['ESM-MSR_qkv_only_2'] = df_esm_qkv_only_2['combined_pred']
    try:
        df_esm_qkv_only_2['ESM-MSR_qkv_only_additive_2'] = df_esm_qkv_only_2['combined_pred_additive']
    except:
        pass

    df_esm_qkv_only_3 = pd.read_csv(f'{qkv_only_base_3}_alpha{int(float(full_alpha))}.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_qkv_only_3['ESM-MSR_qkv_only_3'] = df_esm_qkv_only_3['combined_pred']
    try:
        df_esm_qkv_only_3['ESM-MSR_qkv_only_additive_3'] = df_esm_qkv_only_3['combined_pred_additive']
    except:
        pass

    df_esm_qkv_only = pd.concat([df_esm_qkv_only_1, df_esm_qkv_only_2[[c for c in df_esm_qkv_only_2.columns if 'MSR' in c]], df_esm_qkv_only_3[[c for c in df_esm_qkv_only_3.columns if 'MSR' in c]]], axis=1)

    df_esm_medmed_1 = pd.read_csv(f'{medmed_base_1}_alpha8.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_medmed_1['ESM-MSR_medmed_1'] = df_esm_medmed_1['combined_pred']
    try:
        df_esm_medmed_1['ESM-MSR_medmed_additive_1'] = df_esm_medmed_1['combined_pred_additive']
    except:
        pass
    df_esm_medmed_1['ddG_ML_esm'] = df_esm_medmed_1[truth]

    df_esm_medmed_2 = pd.read_csv(f'{medmed_base_2}_alpha8.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_medmed_2['ESM-MSR_medmed_2'] = df_esm_medmed_2['combined_pred']
    try:
        df_esm_medmed_2['ESM-MSR_medmed_additive_2'] = df_esm_medmed_2['combined_pred_additive']
    except:
        pass

    df_esm_medmed_3 = pd.read_csv(f'{medmed_base_3}_alpha8.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_medmed_3['ESM-MSR_medmed_3'] = df_esm_medmed_3['combined_pred']
    try:
        df_esm_medmed_3['ESM-MSR_medmed_additive_3'] = df_esm_medmed_3['combined_pred_additive']
    except:
        pass

    df_esm_medmed = pd.concat([df_esm_medmed_1, df_esm_medmed_2[[c for c in df_esm_medmed_2.columns if 'MSR' in c]], df_esm_medmed_3[[c for c in df_esm_medmed_3.columns if 'MSR' in c]]], axis=1)

    df_esm_detach_reg_1 = pd.read_csv(f'{detach_reg_base_1}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_detach_reg_1['ESM-MSR_detach_reg_1'] = df_esm_detach_reg_1['combined_pred']
    try:
        df_esm_detach_reg_1['ESM-MSR_detach_reg_additive_1'] = df_esm_detach_reg_1['combined_pred_additive']
    except:
        pass
    df_esm_detach_reg_1['ddG_ML_esm'] = df_esm_detach_reg_1[truth]

    df_esm_detach_reg_2 = pd.read_csv(f'{detach_reg_base_2}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_detach_reg_2['ESM-MSR_detach_reg_2'] = df_esm_detach_reg_2['combined_pred']
    try:
        df_esm_detach_reg_2['ESM-MSR_detach_reg_additive_2'] = df_esm_detach_reg_2['combined_pred_additive']
    except:
        pass

    df_esm_detach_reg_3 = pd.read_csv(f'{detach_reg_base_3}_alpha{int(float(full_alpha))}.0_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_detach_reg_3['ESM-MSR_detach_reg_3'] = df_esm_detach_reg_3['combined_pred']
    try:
        df_esm_detach_reg_3['ESM-MSR_detach_reg_additive_3'] = df_esm_detach_reg_3['combined_pred_additive']
    except:
        pass

    df_esm_detach_reg = pd.concat([df_esm_detach_reg_1, df_esm_detach_reg_2[[c for c in df_esm_detach_reg_2.columns if 'MSR' in c]], df_esm_detach_reg_3[[c for c in df_esm_detach_reg_3.columns if 'MSR' in c]]], axis=1)

    df_esm_r1_1 = pd.read_csv(f'{r1_base_1}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_r1_1['ESM-MSR_r1_1'] = df_esm_r1_1['combined_pred']
    try:
        df_esm_r1_1['ESM-MSR_r1_additive_1'] = df_esm_r1_1['combined_pred_additive']
    except:
        pass
    df_esm_r1_1['ddG_ML_esm'] = df_esm_r1_1[truth]

    df_esm_r1_2 = pd.read_csv(f'{r1_base_2}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_r1_2['ESM-MSR_r1_2'] = df_esm_r1_2['combined_pred']
    try:
        df_esm_r1_2['ESM-MSR_r1_additive_2'] = df_esm_r1_2['combined_pred_additive']
    except:
        pass

    df_esm_r1_3 = pd.read_csv(f'{r1_base_3}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_r1_3['ESM-MSR_r1_3'] = df_esm_r1_3['combined_pred']
    try:
        df_esm_r1_3['ESM-MSR_r1_additive_3'] = df_esm_r1_3['combined_pred_additive']
    except:
        pass

    df_esm_r1 = pd.concat([df_esm_r1_1, df_esm_r1_2[[c for c in df_esm_r1_2.columns if 'MSR' in c]], df_esm_r1_3[[c for c in df_esm_r1_3.columns if 'MSR' in c]]], axis=1)

    df_esm_fused_1 = pd.read_csv(f'{r1_base_1}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_fused_1['ESM-MSR_fused_1'] = df_esm_fused_1['combined_pred']
    try:
        df_esm_fused_1['ESM-MSR_fused_additive_1'] = df_esm_fused_1['combined_pred_additive']
    except:
        pass
    df_esm_fused_1['ddG_ML_esm'] = df_esm_fused_1[truth]

    df_esm_fused_2 = pd.read_csv(f'{r1_base_2}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_fused_2['ESM-MSR_fused_2'] = df_esm_fused_2['combined_pred']
    try:
        df_esm_fused_2['ESM-MSR_fused_additive_2'] = df_esm_fused_2['combined_pred_additive']
    except:
        pass

    df_esm_fused_3 = pd.read_csv(f'{r1_base_3}_alpha2.0_unmasked_predictions.csv', index_col=0).rename_axis(index={'id': 'uid'})
    df_esm_fused_3['ESM-MSR_fused_3'] = df_esm_fused_3['combined_pred']
    try:
        df_esm_fused_3['ESM-MSR_fused_additive_3'] = df_esm_fused_3['combined_pred_additive']
    except:
        pass

    df_esm_fused = pd.concat([df_esm_fused_1, df_esm_fused_2[[c for c in df_esm_fused_2.columns if 'MSR' in c]], df_esm_fused_3[[c for c in df_esm_fused_3.columns if 'MSR' in c]]], axis=1)

    # ESM3 Zero Shot
    df_esm_zs = pd.read_csv(f'{base_path}/zeroshot_alpha0.0_unmasked_predictions.csv', index_col=0).sort_index()
    df_esm_zs['ESM3-small-open'] = df_esm_zs['combined_pred']

    try:
        df_esm_zs['ESM3-small-open_additive'] = df_esm_zs['combined_pred_additive']
    except:
        df_esm_zs['ESM3-small-open_additive'] = np.nan
        pass

    # ESM3 Zero Shot
    df_esm_zs_masked = pd.read_csv(f'{base_path}/zeroshot_alpha0.0_marginal_predictions.csv', index_col=0).sort_index()
    df_esm_zs_masked['ESM3-small-open_masked'] = df_esm_zs_masked['combined_pred']

    try:
        df_esm_zs_masked['ESM3-small-open_masked_additive'] = df_esm_zs_masked['combined_pred_additive']
    except:
        df_esm_zs_masked['ESM3-small-open_masked_additive'] = np.nan
        pass

    # ESM3 Zero Shot
    df_esm_zs_chain = pd.read_csv(f'{base_path}/zeroshot_alpha0.0_chain_predictions.csv', index_col=0).sort_index()
    df_esm_zs_chain['ESM3-small-open_chain'] = df_esm_zs_chain['combined_pred']

    try:
        df_esm_zs_chain['ESM3-small-open_chain_additive'] = df_esm_zs_chain['combined_pred_additive']
    except:
        df_esm_zs_chain['ESM3-small-open_chain_additive'] = np.nan
        pass

    df_thermo = deduplicate_and_check(df_thermo, dataset, "df_thermo")
    df_me = deduplicate_and_check(df_me, dataset, "df_me")
    #df_me_single = deduplicate_and_check(df_me_single, dataset, "df_me_single")
    df_ros = deduplicate_and_check(df_ros, dataset, "df_ros")
    df_mpnn = deduplicate_and_check(df_mpnn, dataset, "df_mpnn")
    df_mpnn_masked = deduplicate_and_check(df_mpnn_masked, dataset, "df_mpnn_masked")
    df_spurs = deduplicate_and_check(df_spurs, dataset, "df_spurs")
    df_spurs_reported = deduplicate_and_check(df_spurs_reported, dataset, "df_spurs_reported")
    
    df_esm = deduplicate_and_check(df_esm, dataset, "df_esm")
    df_esm_wt = deduplicate_and_check(df_esm_wt, dataset, "df_esm_wt")
    df_esm_mt = deduplicate_and_check(df_esm_mt, dataset, "df_esm_mt")
    df_esm_defchain = deduplicate_and_check(df_esm_defchain, dataset, "df_esm_defchain")
    df_esm_defmarginal = deduplicate_and_check(df_esm_defmarginal, dataset, "df_esm_defmarginal")
    df_esm_alpha_high = deduplicate_and_check(df_esm_alpha_high, dataset, "df_esm_alpha_high")
    df_esm_alpha_med = deduplicate_and_check(df_esm_alpha_med, dataset, "df_esm_alpha_med")
    df_esm_alpha_low = deduplicate_and_check(df_esm_alpha_low, dataset, "df_esm_alpha_low")

    df_esm_chain = deduplicate_and_check(df_esm_chain, dataset, "df_esm_chain")
    df_esm_masked = deduplicate_and_check(df_esm_masked, dataset, "df_esm_masked")
    df_esm_single = deduplicate_and_check(df_esm_single, dataset, "df_esm_single")
    df_esm_noseqhead = deduplicate_and_check(df_esm_noseqhead, dataset, "df_esm_noseqhead")
    df_esm_all = deduplicate_and_check(df_esm_all, dataset, "df_esm_all")
    df_esm_ffn = deduplicate_and_check(df_esm_ffn, dataset, "df_esm_ffn")
    df_esm_qkv_outproj = deduplicate_and_check(df_esm_qkv_outproj, dataset, "df_esm_qkv_outproj")
    df_esm_noqkv = deduplicate_and_check(df_esm_noqkv, dataset, "df_esm_noqkv")
    df_esm_qkv_only = deduplicate_and_check(df_esm_qkv_only, dataset, "df_esm_qkv_only")
    df_esm_medmed = deduplicate_and_check(df_esm_medmed, dataset, "df_esm_medmed")
    df_esm_detach_reg = deduplicate_and_check(df_esm_detach_reg, dataset, "df_esm_detach_reg")
    df_esm_r1 = deduplicate_and_check(df_esm_r1, dataset, "df_esm_r1")
    df_esm_fused = deduplicate_and_check(df_esm_fused, dataset, "df_esm_fused")

    df_esm_zs = deduplicate_and_check(df_esm_zs, dataset, "df_esm_zs")
    df_esm_zs_masked = deduplicate_and_check(df_esm_zs_masked, dataset, "df_esm_zs_masked")
    df_esm_zs_chain = deduplicate_and_check(df_esm_zs_chain, dataset, "df_esm_zs_chain")
    
    # Missing from original: Must deduplicate zero-shot to prevent join multiplication
    df_zero_sm = deduplicate_and_check(df_zero_sm, dataset, "df_zero_sm")
    df_zero_med = deduplicate_and_check(df_zero_med, dataset, "df_zero_med")
    df_zero_lg = deduplicate_and_check(df_zero_lg, dataset, "df_zero_lg")

    # ================= FINAL JOIN =================

    df = df_thermo[[c for c in df_thermo.columns if 'ThermoMPNN' in c] + ['ddG_ML_thermo']].join( #, how='left')
        df_me[[c for c in df_me.columns if 'MutateEverything' in c] + ['ddG_ML_me']], how='left').join(
        df_zero_sm[[c for c in df_zero_sm.columns if 'ESM3' in c]], how='left').join(
        df_zero_med[[c for c in df_zero_med.columns if 'ESM3' in c]], how='left').join(
        df_zero_lg[[c for c in df_zero_lg.columns if 'ESM3' in c]], how='left').join(
        df_spurs[[c for c in df_spurs.columns if 'SPURS' in c]], how='left').join(
        df_spurs_reported[[c for c in df_spurs_reported.columns if 'SPURS_reported' in c]], how='left').join(
        df_ros[[c for c in df_ros.columns if 'Rosetta' in c]], how='left').join(
        df_mpnn[[c for c in df_mpnn.columns if 'ProteinMPNN' in c]], how='left').join(
        df_mpnn_masked[[c for c in df_mpnn_masked.columns if 'ProteinMPNN' in c]], how='left'
    )

    df = df.join(df_esm[[c for c in df_esm.columns if 'MSR' in c]+['code', 'mut_type', 'ddG_ML_esm']], how='left')
    df = df.join(df_esm_wt[[c for c in df_esm_wt.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_mt[[c for c in df_esm_mt.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_defchain[[c for c in df_esm_defchain.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_defmarginal[[c for c in df_esm_defmarginal.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_alpha_high[[c for c in df_esm_alpha_high.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_alpha_med[[c for c in df_esm_alpha_med.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_alpha_low[[c for c in df_esm_alpha_low.columns if 'MSR' in c]], how='left')

    df = df.join(df_esm_single[[c for c in df_esm_single.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_reg[[c for c in df_esm_reg.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_masked[[c for c in df_esm_masked.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_chain[[c for c in df_esm_chain.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_noseqhead[[c for c in df_esm_noseqhead.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_all[[c for c in df_esm_all.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_ffn[[c for c in df_esm_ffn.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_qkv_outproj[[c for c in df_esm_qkv_outproj.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_noqkv[[c for c in df_esm_noqkv.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_qkv_only[[c for c in df_esm_qkv_only.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_medmed[[c for c in df_esm_medmed.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_detach_reg[[c for c in df_esm_detach_reg.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_r1[[c for c in df_esm_r1.columns if 'MSR' in c]], how='left')
    df = df.join(df_esm_fused[[c for c in df_esm_fused.columns if 'MSR' in c]], how='left')

    df = df.join(df_esm_zs[['ESM3-small-open', 'ESM3-small-open_additive']], how='left')
    df = df.join(df_esm_zs_masked[['ESM3-small-open_masked', 'ESM3-small-open_masked_additive']], how='left')
    df = df.join(df_esm_zs_chain[['ESM3-small-open_chain', 'ESM3-small-open_chain_additive']], how='left')

    df = df.rename(columns={'ddG_ML_esm': 'ddG'})

    # Estimated measurement error and noise for Tsuboyama analysis
    df['random_1'] = np.random.rand(len(df))
    df['random_2'] = np.random.rand(len(df))
    df['random_3'] = np.random.rand(len(df))

    df['ddG_1'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))
    df['ddG_2'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))
    df['ddG_3'] = df['ddG'] + np.random.normal(0, 0.14, size=len(df))

    results[dataset] = df

#4 minutes

domainome
Info: df_thermo dropped irrelevant columns before aggregation: ['WT_name', 'batch']
Info: df_thermo ('domainome') length went from 784909 to 572204 after deduplication.
Info: df_me ('domainome') length went from 536145 to 536145 after deduplication.
Info: df_mpnn dropped irrelevant columns before aggregation: ['mpnn_score_mut_seq', 'runtime', 'mpnn_score_epistasis', 'pdb_file', 'uniprot_ID', 'mpnn_score_wt_seq']
Info: df_mpnn ('domainome') length went from 576403 to 572204 after deduplication.
Info: df_mpnn_masked dropped irrelevant columns before aggregation: ['runtime', 'mpnn_score_epistasis', 'pdb_file', 'uniprot_ID']
Info: df_mpnn_masked ('domainome') length went from 576403 to 572204 after deduplication.
Info: df_spurs dropped irrelevant columns before aggregation: ['pdb_file', 'uniprot_ID']
Info: df_spurs ('domainome') length went from 576403 to 572204 after deduplication.
Info: df_spurs_reported dropped irrelevant columns before aggregation: ['pdb_file', 'uniprot_ID']


In [3]:
models=('ESM-MSR_1', 'ESM-MSR_2', 'ESM-MSR_3',
        #'ESM-MSR_WT_additive_1', 'ESM-MSR_WT_additive_2', 'ESM-MSR_WT_additive_3',
        'ESM-MSR_WT_1', 'ESM-MSR_WT_2', 'ESM-MSR_WT_3',
        'ESM-MSR_MT_1', 'ESM-MSR_MT_2', 'ESM-MSR_MT_3',
        'ESM-MSR_alpha_high_1', 'ESM-MSR_alpha_high_2', 'ESM-MSR_alpha_high_3',  
        'ESM-MSR_alpha_med_1', 'ESM-MSR_alpha_med_2', 'ESM-MSR_alpha_med_3', 
        'ESM-MSR_alpha_low_1', 'ESM-MSR_alpha_low_2', 'ESM-MSR_alpha_low_3',
        'ESM-MSR_single_1', 'ESM-MSR_single_2', 'ESM-MSR_single_3', 
        'ESM-MSR_reg_1', 'ESM-MSR_reg_2', 'ESM-MSR_reg_3',
        'ESM-MSR_defchain_1', 'ESM-MSR_defchain_2', 'ESM-MSR_defchain_3',
        'ESM-MSR_defmarginal_1', 'ESM-MSR_defmarginal_2', 'ESM-MSR_defmarginal_3',
        'ESM-MSR_masked_1', 'ESM-MSR_masked_2', 'ESM-MSR_masked_3',
        'ESM-MSR_chain_1', 'ESM-MSR_chain_2', 'ESM-MSR_chain_3', 
        'ESM-MSR_noseqhead_1', 'ESM-MSR_noseqhead_2', 'ESM-MSR_noseqhead_3', 
        'ESM-MSR_all_1', 'ESM-MSR_all_2', 'ESM-MSR_all_3', 
        'ESM-MSR_ffn_1', 'ESM-MSR_ffn_2', 'ESM-MSR_ffn_3', 
        'ESM-MSR_qkv_outproj_1', 'ESM-MSR_qkv_outproj_2', 'ESM-MSR_qkv_outproj_3', 
        'ESM-MSR_noqkv_1', 'ESM-MSR_noqkv_2', 'ESM-MSR_noqkv_3', 
        'ESM-MSR_qkv_only_1', 'ESM-MSR_qkv_only_2', 'ESM-MSR_qkv_only_3',
        'ESM-MSR_medmed_1', 'ESM-MSR_medmed_2', 'ESM-MSR_medmed_3', 
        'ESM-MSR_detach_reg_1', 'ESM-MSR_detach_reg_2', 'ESM-MSR_detach_reg_3', 
        'ESM-MSR_r1_1', 'ESM-MSR_r1_2', 'ESM-MSR_r1_3',
        'ESM-MSR_fused_1', 'ESM-MSR_fused_2', 'ESM-MSR_fused_3',
        'SPURS_1', 'SPURS_2', 'SPURS_3',
        'SPURS_additive_1', 'SPURS_additive_2', 'SPURS_additive_3',
        'SPURS_reported', 'SPURS_reported_additive',
        'ThermoMPNN_1', 'ThermoMPNN_2', 'ThermoMPNN_3', 
        #'ThermoMPNN(-D)_1', 'ThermoMPNN(-D)_2', 'ThermoMPNN(-D)_3', 
        'MutateEverything_1', 'MutateEverything_2', 'MutateEverything_3',
        'MutateEverything_additive_1', 'MutateEverything_additive_2', 'MutateEverything_additive_3',
        'ESM3-small-open', 'ESM3-small-open_masked', 'ESM3-small-open_chain',
        'ESM3-small', 'ESM3-medium', 'ESM3-large', 
        #'Rosetta Cartesian DDG_1', 'Rosetta Cartesian DDG_2', 'Rosetta Cartesian DDG_3', 
        'ProteinMPNN', 'ProteinMPNN_masked')

In [4]:
splits=tuple(['domainome'])

df_scores_domainome = pd.DataFrame(index=pd.MultiIndex.from_product([['domainome'], ['ungrouped', 'grouped']]))

df_scores_domainome = assess_all_models(results['domainome'].dropna(axis=1, how='all'), df_scores_domainome, assess_grouped_spearman, splits=splits, models=models)

df_ddg_domainome = df_scores_domainome[[c for c in df_scores_domainome.columns if not c.endswith('_n')]].T
df_ddg_domainome = df_ddg_domainome.loc[~df_ddg_domainome.index.str.contains('groups')]
df_ddg_domainome

ESM-MSR_1
ESM-MSR_2
ESM-MSR_3
ESM-MSR_WT_1
ESM-MSR_WT_2
ESM-MSR_WT_3
ESM-MSR_MT_1
ESM-MSR_MT_2
ESM-MSR_MT_3
ESM-MSR_alpha_high_1
ESM-MSR_alpha_high_2
ESM-MSR_alpha_high_3
ESM-MSR_alpha_med_1
ESM-MSR_alpha_med_2
ESM-MSR_alpha_med_3
ESM-MSR_alpha_low_1
ESM-MSR_alpha_low_2
ESM-MSR_alpha_low_3
ESM-MSR_single_1
ESM-MSR_single_2
ESM-MSR_single_3
ESM-MSR_reg_1
ESM-MSR_reg_2
ESM-MSR_reg_3
ESM-MSR_defchain_1
ESM-MSR_defchain_2
ESM-MSR_defchain_3
ESM-MSR_defmarginal_1
ESM-MSR_defmarginal_2
ESM-MSR_defmarginal_3
ESM-MSR_masked_1
ESM-MSR_masked_2
ESM-MSR_masked_3
ESM-MSR_chain_1
ESM-MSR_chain_2
ESM-MSR_chain_3
ESM-MSR_noseqhead_1
ESM-MSR_noseqhead_2
ESM-MSR_noseqhead_3
ESM-MSR_all_1
ESM-MSR_all_2
ESM-MSR_all_3
ESM-MSR_ffn_1
ESM-MSR_ffn_2
ESM-MSR_ffn_3
ESM-MSR_qkv_outproj_1
ESM-MSR_qkv_outproj_2
ESM-MSR_qkv_outproj_3
ESM-MSR_noqkv_1
ESM-MSR_noqkv_2
ESM-MSR_noqkv_3
ESM-MSR_qkv_only_1
ESM-MSR_qkv_only_2
ESM-MSR_qkv_only_3
ESM-MSR_medmed_1
ESM-MSR_medmed_2
ESM-MSR_medmed_3
ESM-MSR_detach_reg_1
ESM-MSR

domainome          
                   ungrouped   grouped
ESM-MSR_1           0.544808  0.547073
ESM-MSR_2           0.544437  0.547484
ESM-MSR_3           0.547723  0.552074
ESM-MSR_WT_1        0.542909  0.544331
ESM-MSR_WT_2        0.543283  0.546064
...                      ...       ...
ESM3-small          0.528891  0.547119
ESM3-medium         0.503326  0.531377
ESM3-large          0.455502  0.507536
ProteinMPNN         0.530813  0.547454
ProteinMPNN_masked  0.530908  0.547649

[91 rows x 2 columns]

In [5]:
df_scores_domainome

ESM-MSR_1    ESM-MSR_1_n  ESM-MSR_1_groups  ESM-MSR_2  \
domainome ungrouped   0.544808  540265.000000               NaN   0.544437   
          grouped     0.547073    1034.990421             522.0   0.547484   

                       ESM-MSR_2_n  ESM-MSR_2_groups  ESM-MSR_3  \
domainome ungrouped  540265.000000               NaN   0.547723   
          grouped      1034.990421             522.0   0.552074   

                       ESM-MSR_3_n  ESM-MSR_3_groups  ESM-MSR_WT_1  ...  \
domainome ungrouped  540265.000000               NaN      0.542909  ...   
          grouped      1034.990421             522.0      0.544331  ...   

                     ESM3-medium_groups  ESM3-large   ESM3-large_n  \
domainome ungrouped                 NaN    0.455502  540265.000000   
          grouped                 522.0    0.507536    1034.990421   

                     ESM3-large_groups  ProteinMPNN  ProteinMPNN_n  \
domainome ungrouped                NaN     0.530813  540265.000000   
          grouped                522.0     0.547454    1034.990421   

                     ProteinMPNN_groups  ProteinMPNN_masked  \
domainome ungrouped                 NaN            0.530908   
          grouped                 522.0            0.547649   

                     ProteinMPNN_masked_n  ProteinMPNN_masked_groups  
domainome ungrouped         540265.000000                        NaN  
          grouped             1034.990421                      522.0  

[2 rows x 273 columns]

In [6]:
results['hyperopt_splits-test']['Rosetta Cartesian DDG_additive_1']

uid
1UFM_A11C                -0.793197
1UFM_A11D                -0.587075
1UFM_A11E                 0.196939
1UFM_A11F                 0.565646
1UFM_A11G                -1.001361
                            ...   
HHH_rd1_0949_R10Y:E36S    0.812925
HHH_rd1_0949_R10Y:E36T    0.812925
HHH_rd1_0949_R10Y:E36V    0.812925
HHH_rd1_0949_R10Y:E36W    0.812925
HHH_rd1_0949_R10Y:E36Y    0.812925
Name: Rosetta Cartesian DDG_additive_1, Length: 43700, dtype: float64

In [7]:
models=('ESM-MSR_1', 'ESM-MSR_2', 'ESM-MSR_3',
        'ESM-MSR_WT_additive_1', 'ESM-MSR_WT_additive_2', 'ESM-MSR_WT_additive_3',
        'ESM-MSR_WT_1', 'ESM-MSR_WT_2', 'ESM-MSR_WT_3',
        'ESM-MSR_MT_1', 'ESM-MSR_MT_2', 'ESM-MSR_MT_3',
        'ESM-MSR_alpha_high_1', 'ESM-MSR_alpha_high_2', 'ESM-MSR_alpha_high_3',  
        'ESM-MSR_alpha_med_1', 'ESM-MSR_alpha_med_2', 'ESM-MSR_alpha_med_3', 
        'ESM-MSR_alpha_low_1', 'ESM-MSR_alpha_low_2', 'ESM-MSR_alpha_low_3',
        'ESM-MSR_single_1', 'ESM-MSR_single_2', 'ESM-MSR_single_3', 
        'ESM-MSR_reg_1', 'ESM-MSR_reg_2', 'ESM-MSR_reg_3',
        'ESM-MSR_defchain_1', 'ESM-MSR_defchain_2', 'ESM-MSR_defchain_3',
        'ESM-MSR_defmarginal_1', 'ESM-MSR_defmarginal_2', 'ESM-MSR_defmarginal_3',
        'ESM-MSR_masked_1', 'ESM-MSR_masked_2', 'ESM-MSR_masked_3',
        'ESM-MSR_chain_1', 'ESM-MSR_chain_2', 'ESM-MSR_chain_3', 
        'ESM-MSR_noseqhead_1', 'ESM-MSR_noseqhead_2', 'ESM-MSR_noseqhead_3', 
        'ESM-MSR_all_1', 'ESM-MSR_all_2', 'ESM-MSR_all_3', 
        'ESM-MSR_ffn_1', 'ESM-MSR_ffn_2', 'ESM-MSR_ffn_3', 
        'ESM-MSR_qkv_outproj_1', 'ESM-MSR_qkv_outproj_2', 'ESM-MSR_qkv_outproj_3', 
        'ESM-MSR_noqkv_1', 'ESM-MSR_noqkv_2', 'ESM-MSR_noqkv_3', 
        'ESM-MSR_qkv_only_1', 'ESM-MSR_qkv_only_2', 'ESM-MSR_qkv_only_3', 
        'ESM-MSR_medmed_1', 'ESM-MSR_medmed_2', 'ESM-MSR_medmed_3', 
        'ESM-MSR_detach_reg_1', 'ESM-MSR_detach_reg_2', 'ESM-MSR_detach_reg_3', 
        'ESM-MSR_r1_1', 'ESM-MSR_r1_2', 'ESM-MSR_r1_3',
        'ESM-MSR_fused_1', 'ESM-MSR_fused_2', 'ESM-MSR_fused_3',
        'SPURS_1', 'SPURS_2', 'SPURS_3',
        'SPURS_additive_1', 'SPURS_additive_2', 'SPURS_additive_3',
        'SPURS_reported', 'SPURS_reported_additive',
        'ThermoMPNN(-D)_additive_1', 'ThermoMPNN(-D)_additive_2', 'ThermoMPNN(-D)_additive_3', 
        'ThermoMPNN(-D)_1', 'ThermoMPNN(-D)_2', 'ThermoMPNN(-D)_3', 
        'MutateEverything_1', 'MutateEverything_2', 'MutateEverything_3',
        'MutateEverything_additive_1', 'MutateEverything_additive_2', 'MutateEverything_additive_3',
        'ESM3-small-open', 'ESM3-small-open_masked', 'ESM3-small-open_chain',
        'ESM3-small', 'ESM3-medium', 'ESM3-large', 
        'Rosetta Cartesian DDG_1', 'Rosetta Cartesian DDG_2', 'Rosetta Cartesian DDG_3',
        'Rosetta Cartesian DDG_additive_1', 'Rosetta Cartesian DDG_additive_2', 'Rosetta Cartesian DDG_additive_3',
        'ProteinMPNN', 'ProteinMPNN_masked')

In [8]:
splits=('hyperopt_splits-S-test', 'hyperopt_splits-D-test', 'hyperopt_splits-test')

df_scores_hyperopt_test = pd.DataFrame(index=pd.MultiIndex.from_product([['hyperopt_splits-S-test', 'hyperopt_splits-D-test', 'hyperopt_splits-test'], ['ungrouped', 'grouped']]))

df_scores_hyperopt_test = assess_all_models(results['hyperopt_splits-test'].dropna(axis=1, how='all'), df_scores_hyperopt_test, assess_grouped_spearman, splits=splits, models=models)

df_ddg_t = df_scores_hyperopt_test[[c for c in df_scores_hyperopt_test.columns if not c.endswith('_n')]].T
df_ddg_t = df_ddg_t.loc[~df_ddg_t.index.str.contains('groups')]

ESM-MSR_1
ESM-MSR_2
ESM-MSR_3
ESM-MSR_WT_additive_1
ESM-MSR_WT_additive_2
ESM-MSR_WT_additive_3
ESM-MSR_WT_1
ESM-MSR_WT_2
ESM-MSR_WT_3
ESM-MSR_MT_1
ESM-MSR_MT_2
ESM-MSR_MT_3
ESM-MSR_alpha_high_1
ESM-MSR_alpha_high_2
ESM-MSR_alpha_high_3
ESM-MSR_alpha_med_1
ESM-MSR_alpha_med_2
ESM-MSR_alpha_med_3
ESM-MSR_alpha_low_1
ESM-MSR_alpha_low_2
ESM-MSR_alpha_low_3
ESM-MSR_single_1
ESM-MSR_single_2
ESM-MSR_single_3
ESM-MSR_reg_1
ESM-MSR_reg_2
ESM-MSR_reg_3
ESM-MSR_defchain_1
ESM-MSR_defchain_2
ESM-MSR_defchain_3
ESM-MSR_defmarginal_1
ESM-MSR_defmarginal_2
ESM-MSR_defmarginal_3
ESM-MSR_masked_1
ESM-MSR_masked_2
ESM-MSR_masked_3
ESM-MSR_chain_1
ESM-MSR_chain_2
ESM-MSR_chain_3
ESM-MSR_noseqhead_1
ESM-MSR_noseqhead_2
ESM-MSR_noseqhead_3
ESM-MSR_all_1
ESM-MSR_all_2
ESM-MSR_all_3
ESM-MSR_ffn_1
ESM-MSR_ffn_2
ESM-MSR_ffn_3
ESM-MSR_qkv_outproj_1
ESM-MSR_qkv_outproj_2
ESM-MSR_qkv_outproj_3
ESM-MSR_noqkv_1
ESM-MSR_noqkv_2
ESM-MSR_noqkv_3
ESM-MSR_qkv_only_1
ESM-MSR_qkv_only_2
ESM-MSR_qkv_only_3
ESM-MSR_medme

In [9]:
splits=('hyperopt_splits-S-val', 'hyperopt_splits-D-val', 'hyperopt_splits-val')

df_scores_hyperopt_val = pd.DataFrame(index=pd.MultiIndex.from_product([['hyperopt_splits-S-val', 'hyperopt_splits-D-val', 'hyperopt_splits-val'], ['ungrouped', 'grouped']]))

df_scores_hyperopt_val = assess_all_models(results['hyperopt_splits-val'].dropna(axis=1, how='all'), df_scores_hyperopt_val, assess_grouped_spearman, splits=splits, models=models)

df_ddg_v = df_scores_hyperopt_val[[c for c in df_scores_hyperopt_val.columns if not c.endswith('_n')]].T
df_ddg_v = df_ddg_v.loc[~df_ddg_v.index.str.contains('groups')]

ESM-MSR_1
ESM-MSR_2
ESM-MSR_3
ESM-MSR_WT_additive_1
ESM-MSR_WT_additive_2
ESM-MSR_WT_additive_3
ESM-MSR_WT_1
ESM-MSR_WT_2
ESM-MSR_WT_3
ESM-MSR_MT_1
ESM-MSR_MT_2
ESM-MSR_MT_3
ESM-MSR_alpha_high_1
ESM-MSR_alpha_high_2
ESM-MSR_alpha_high_3
ESM-MSR_alpha_med_1
ESM-MSR_alpha_med_2
ESM-MSR_alpha_med_3
ESM-MSR_alpha_low_1
ESM-MSR_alpha_low_2
ESM-MSR_alpha_low_3
ESM-MSR_single_1
ESM-MSR_single_2
ESM-MSR_single_3
ESM-MSR_reg_1
ESM-MSR_reg_2
ESM-MSR_reg_3
ESM-MSR_defchain_1
ESM-MSR_defchain_2
ESM-MSR_defchain_3
ESM-MSR_defmarginal_1
ESM-MSR_defmarginal_2
ESM-MSR_defmarginal_3
ESM-MSR_masked_1
ESM-MSR_masked_2
ESM-MSR_masked_3
ESM-MSR_chain_1
ESM-MSR_chain_2
ESM-MSR_chain_3
ESM-MSR_noseqhead_1
ESM-MSR_noseqhead_2
ESM-MSR_noseqhead_3
ESM-MSR_all_1
ESM-MSR_all_2
ESM-MSR_all_3
ESM-MSR_ffn_1
ESM-MSR_ffn_2
ESM-MSR_ffn_3
ESM-MSR_qkv_outproj_1
ESM-MSR_qkv_outproj_2
ESM-MSR_qkv_outproj_3
ESM-MSR_noqkv_1
ESM-MSR_noqkv_2
ESM-MSR_noqkv_3
ESM-MSR_qkv_only_1
ESM-MSR_qkv_only_2
ESM-MSR_qkv_only_3
ESM-MSR_medme

In [10]:
import pandas as pd
import numpy as np
import re

def collapse_replicates(df):
    """
    Takes a dataframe where the index contains replicates with suffixes (e.g., _1, _2).
    Groups them by the base name and replaces the values with 'mean ± std'.
    """
    # Ensure the input is a copy to avoid SettingWithCopy warnings on the original
    df = df.copy()

    # 1. Identify the grouping key (Base Name)
    # We use regex to remove '_\d+' (underscore followed by digits) from the end of the string
    # If the index is not currently set, we assume the first column is the identifier
    if df.index.name is None and not df.index.is_object():
         df = df.set_index(df.columns[0])
    
    # Extract base names (e.g., "ESM-MSR_1" -> "ESM-MSR")
    base_names = df.index.astype(str).str.replace(r'_\d+$', '', regex=True)

    # 2. Group by the base name and calculate mean and std
    grouped = df.groupby(base_names)
    means = grouped.mean()
    stds = grouped.std()

    # 3. Format the output columns
    # We create a new DataFrame to store the string results
    result_df = pd.DataFrame(index=means.index, columns=means.columns)

    # Iterate through columns to format as "mean ± std"
    for col in means.columns:
        m = means[col]
        s = stds[col]
        
        # Combine mean and std. 
        # We fill NaN stds with 0 (happens if there was only 1 replicate)
        # You can adjust the {:.6f} to change the decimal precision
        result_df[col] = [
            f"{val_m:.3f} ± {val_s:.3f}" if not np.isnan(val_s) else f"{val_m:.6f}"
            for val_m, val_s in zip(m, s)
        ]

    return result_df

In [ ]:
results_dddg = {}

for dataset, df in results.items():
    if dataset == 'domainome':
        continue

    cols = ['ddG']
    df_epistatic = df.copy(deep=True)

    #print(len(df.loc[df['mut_type'].str.contains(':')]))
    for c in cols:
        if not c.endswith('_n'):
            print(c)
            # synthesize additive mutation predictions and scores
            df_epistatic = sum_individual_mutation_scores(df_epistatic, c, new_score_column=None) #, df_missing, missing1, missing2

    #df_epistatic = df_epistatic[[c for c in df_epistatic.columns if 'additive' in c] + ['code', 'ddG', 'mut_type']]
    df_epistatic = df_epistatic[[c for c in df_epistatic.columns if not c.endswith('_n')]]
    df_epistatic = df_epistatic.dropna(subset='ESM-MSR_1')

    df_dddG = df_epistatic.copy(deep=True)
    df_dddG = compute_dddg(df_dddG)
    df_dddG = df_dddG[[c for c in df_dddG.columns if 'dddG' in c]+['code', 'mut_type']]
    df_dddG.columns = [c[:-5] if c.endswith('dddG') else c for c in df_dddG.columns]
    results_dddg[dataset] = df_dddG

# 3 minutes

ddG
ddG


In [12]:
results_dddg['hyperopt_splits-test']

,ThermoMPNN(-D)_1,ThermoMPNN(-D)_2,ThermoMPNN(-D)_3,MutateEverything_1,MutateEverything_2,MutateEverything_3,ESM3-small,ESM3-medium,ESM3-large,SPURS_1,...,ESM-MSR_r1_3,ESM-MSR_fused_1,ESM-MSR_fused_2,ESM-MSR_fused_3,ESM3-small-open,ESM3-small-open_masked,ESM3-small-open_chain,ddG,code,mut_type
uid,,,,,,,,,,,,,,,,,,,,,
1UFM_A11C,0.000000,0.000000,0.000000,0.222313,0.407803,0.234657,NaN,NaN,NaN,0.044629,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11C
1UFM_A11D,0.000000,0.000000,0.000000,0.020686,-0.019115,0.027772,NaN,NaN,NaN,-0.620703,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11D
1UFM_A11E,0.000000,0.000000,0.000000,0.024526,0.019546,0.027984,NaN,NaN,NaN,-0.688227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11E
1UFM_A11F,0.000000,0.000000,0.000000,0.152340,0.024149,0.005970,NaN,NaN,NaN,0.233623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11F
1UFM_A11G,0.000000,0.000000,0.000000,-0.015866,-0.077159,-0.054052,NaN,NaN,NaN,-0.387187,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HHH_rd1_0949_R10Y:E36S,1.129323,0.125241,0.722613,0.301761,0.119767,0.365533,3.019897,3.752441,1.562500,-0.004037,...,0.761952,0.736344,0.683465,0.761952,0.426094,0.702429,0.557130,0.946647,HHH_rd1_0949,R10Y:E36S
HHH_rd1_0949_R10Y:E36T,0.817832,0.087153,1.086862,0.170535,0.189227,0.270776,2.447266,3.185547,1.265625,-0.179011,...,0.635995,0.669679,0.614768,0.635995,0.231028,0.623472,0.464548,0.602194,HHH_rd1_0949,R10Y:E36T
HHH_rd1_0949_R10Y:E36V,0.250903,0.030070,0.813771,0.559057,0.358084,0.365038,2.771484,3.345703,1.218750,-0.453831,...,1.000854,0.967795,0.897823,1.000854,0.338372,0.805383,0.570938,0.544439,HHH_rd1_0949,R10Y:E36V


In [13]:
models=('ESM-MSR_1', 'ESM-MSR_2', 'ESM-MSR_3',
        #'ESM-MSR_WT_additive_1', 'ESM-MSR_WT_additive_2', 'ESM-MSR_WT_additive_3',
        'ESM-MSR_WT_1', 'ESM-MSR_WT_2', 'ESM-MSR_WT_3',
        'ESM-MSR_MT_1', 'ESM-MSR_MT_2', 'ESM-MSR_MT_3',
        'ESM-MSR_alpha_high_1', 'ESM-MSR_alpha_high_2', 'ESM-MSR_alpha_high_3',  
        'ESM-MSR_alpha_med_1', 'ESM-MSR_alpha_med_2', 'ESM-MSR_alpha_med_3', 
        'ESM-MSR_alpha_low_1', 'ESM-MSR_alpha_low_2', 'ESM-MSR_alpha_low_3',
        'ESM-MSR_single_1', 'ESM-MSR_single_2', 'ESM-MSR_single_3', 
        'ESM-MSR_reg_1', 'ESM-MSR_reg_2', 'ESM-MSR_reg_3',
        'ESM-MSR_defchain_1', 'ESM-MSR_defchain_2', 'ESM-MSR_defchain_3',
        'ESM-MSR_defmarginal_1', 'ESM-MSR_defmarginal_2', 'ESM-MSR_defmarginal_3',
        'ESM-MSR_masked_1', 'ESM-MSR_masked_2', 'ESM-MSR_masked_3',
        'ESM-MSR_chain_1', 'ESM-MSR_chain_2', 'ESM-MSR_chain_3', 
        'ESM-MSR_noseqhead_1', 'ESM-MSR_noseqhead_2', 'ESM-MSR_noseqhead_3', 
        'ESM-MSR_noqkv_1', 'ESM-MSR_noqkv_2', 'ESM-MSR_noqkv_3',
        'ESM-MSR_all_1', 'ESM-MSR_all_2', 'ESM-MSR_all_3', 
        'ESM-MSR_ffn_1', 'ESM-MSR_ffn_2', 'ESM-MSR_ffn_3', 
        'ESM-MSR_qkv_outproj_1', 'ESM-MSR_qkv_outproj_2', 'ESM-MSR_qkv_outproj_3', 
        'ESM-MSR_qkv_only_1', 'ESM-MSR_qkv_only_2', 'ESM-MSR_qkv_only_3',
        'ESM-MSR_medmed_1', 'ESM-MSR_medmed_2', 'ESM-MSR_medmed_3', 
        'ESM-MSR_detach_reg_1', 'ESM-MSR_detach_reg_2', 'ESM-MSR_detach_reg_3', 
        'ESM-MSR_r1_1', 'ESM-MSR_r1_2', 'ESM-MSR_r1_3',
        'ESM-MSR_fused_1', 'ESM-MSR_fused_2', 'ESM-MSR_fused_3',
        'SPURS_1', 'SPURS_2', 'SPURS_3',
        'SPURS_reported', #'SPURS_reported_additive',
        #'SPURS_additive_1', 'SPURS_additive_2', 'SPURS_additive_3',
        #'ThermoMPNN(-D)_additive_1', 'ThermoMPNN(-D)_additive_2', 'ThermoMPNN(-D)_additive_3', 
        'ThermoMPNN(-D)_1', 'ThermoMPNN(-D)_2', 'ThermoMPNN(-D)_3', 
        'MutateEverything_1', 'MutateEverything_2', 'MutateEverything_3',
        #'MutateEverything_additive_1', 'MutateEverything_additive_2', 'MutateEverything_additive_3',
        'ESM3-small-open', 'ESM3-small-open_masked', 'ESM3-small-open_chain',
        'ESM3-small', 'ESM3-medium', 'ESM3-large', 
        'Rosetta Cartesian DDG_1', 'Rosetta Cartesian DDG_2', 'Rosetta Cartesian DDG_3', 
        'ProteinMPNN', 'ProteinMPNN_masked')

In [14]:
results_dddg['hyperopt_splits-test']

,ThermoMPNN(-D)_1,ThermoMPNN(-D)_2,ThermoMPNN(-D)_3,MutateEverything_1,MutateEverything_2,MutateEverything_3,ESM3-small,ESM3-medium,ESM3-large,SPURS_1,...,ESM-MSR_r1_3,ESM-MSR_fused_1,ESM-MSR_fused_2,ESM-MSR_fused_3,ESM3-small-open,ESM3-small-open_masked,ESM3-small-open_chain,ddG,code,mut_type
uid,,,,,,,,,,,,,,,,,,,,,
1UFM_A11C,0.000000,0.000000,0.000000,0.222313,0.407803,0.234657,NaN,NaN,NaN,0.044629,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11C
1UFM_A11D,0.000000,0.000000,0.000000,0.020686,-0.019115,0.027772,NaN,NaN,NaN,-0.620703,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11D
1UFM_A11E,0.000000,0.000000,0.000000,0.024526,0.019546,0.027984,NaN,NaN,NaN,-0.688227,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11E
1UFM_A11F,0.000000,0.000000,0.000000,0.152340,0.024149,0.005970,NaN,NaN,NaN,0.233623,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11F
1UFM_A11G,0.000000,0.000000,0.000000,-0.015866,-0.077159,-0.054052,NaN,NaN,NaN,-0.387187,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1UFM,A11G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HHH_rd1_0949_R10Y:E36S,1.129323,0.125241,0.722613,0.301761,0.119767,0.365533,3.019897,3.752441,1.562500,-0.004037,...,0.761952,0.736344,0.683465,0.761952,0.426094,0.702429,0.557130,0.946647,HHH_rd1_0949,R10Y:E36S
HHH_rd1_0949_R10Y:E36T,0.817832,0.087153,1.086862,0.170535,0.189227,0.270776,2.447266,3.185547,1.265625,-0.179011,...,0.635995,0.669679,0.614768,0.635995,0.231028,0.623472,0.464548,0.602194,HHH_rd1_0949,R10Y:E36T
HHH_rd1_0949_R10Y:E36V,0.250903,0.030070,0.813771,0.559057,0.358084,0.365038,2.771484,3.345703,1.218750,-0.453831,...,1.000854,0.967795,0.897823,1.000854,0.338372,0.805383,0.570938,0.544439,HHH_rd1_0949,R10Y:E36V


In [24]:
splits=tuple(['hyperopt_splits-D-val'])
df_dddg_scores_hyperopt_val = pd.DataFrame(index=pd.MultiIndex.from_product([['hyperopt_splits-D-val'], ['ungrouped', 'grouped']]))

df_dddg_scores_hyperopt_val = assess_all_models(results_dddg['hyperopt_splits-val'].dropna(axis=1, how='all'), df_dddg_scores_hyperopt_val, assess_grouped_spearman, splits=splits, models=models)

df_dddg_v = df_dddg_scores_hyperopt_val[[c for c in df_dddg_scores_hyperopt_val.columns if not c.endswith('_n')]].T
df_dddg_v = df_dddg_v.loc[~df_dddg_v.index.str.contains('groups')]
df_dddg_v.columns = pd.MultiIndex.from_tuples([('hyperopt_splits-D-val-dddG', 'ungrouped'), ('hyperopt_splits-D-val-dddG', 'grouped')])

ESM-MSR_1
ESM-MSR_2
ESM-MSR_3
ESM-MSR_WT_1
ESM-MSR_WT_2
ESM-MSR_WT_3
ESM-MSR_MT_1
ESM-MSR_MT_2
ESM-MSR_MT_3
ESM-MSR_alpha_high_1
ESM-MSR_alpha_high_2
ESM-MSR_alpha_high_3
ESM-MSR_alpha_med_1
ESM-MSR_alpha_med_2
ESM-MSR_alpha_med_3
ESM-MSR_alpha_low_1
ESM-MSR_alpha_low_2
ESM-MSR_alpha_low_3
ESM-MSR_single_1
ESM-MSR_single_2
ESM-MSR_single_3
ESM-MSR_reg_1
ESM-MSR_reg_2
ESM-MSR_reg_3
ESM-MSR_defchain_1
ESM-MSR_defchain_2
ESM-MSR_defchain_3
ESM-MSR_defmarginal_1
ESM-MSR_defmarginal_2
ESM-MSR_defmarginal_3
ESM-MSR_masked_1
ESM-MSR_masked_2
ESM-MSR_masked_3
ESM-MSR_chain_1
ESM-MSR_chain_2
ESM-MSR_chain_3
ESM-MSR_noseqhead_1
ESM-MSR_noseqhead_2
ESM-MSR_noseqhead_3
ESM-MSR_noqkv_1
ESM-MSR_noqkv_2
ESM-MSR_noqkv_3
ESM-MSR_all_1
ESM-MSR_all_2
ESM-MSR_all_3
ESM-MSR_ffn_1
ESM-MSR_ffn_2
ESM-MSR_ffn_3
ESM-MSR_qkv_outproj_1
ESM-MSR_qkv_outproj_2
ESM-MSR_qkv_outproj_3
ESM-MSR_qkv_only_1
ESM-MSR_qkv_only_2
ESM-MSR_qkv_only_3
ESM-MSR_medmed_1
ESM-MSR_medmed_2
ESM-MSR_medmed_3
ESM-MSR_detach_reg_1
ESM-MSR

In [15]:
splits=tuple(['hyperopt_splits-D-test'])
df_dddg_scores_hyperopt_test = pd.DataFrame(index=pd.MultiIndex.from_product([['hyperopt_splits-D-test'], ['ungrouped', 'grouped']]))

df_dddg_scores_hyperopt_test = assess_all_models(results_dddg['hyperopt_splits-test'].dropna(axis=1, how='all').dropna(), df_dddg_scores_hyperopt_test, assess_grouped_spearman, splits=splits, models=models)

df_dddg_t = df_dddg_scores_hyperopt_test[[c for c in df_dddg_scores_hyperopt_test.columns if not c.endswith('_n')]].T
df_dddg_t = df_dddg_t.loc[~df_dddg_t.index.str.contains('groups')]
df_dddg_t.columns = pd.MultiIndex.from_tuples([('hyperopt_splits-D-test-dddG', 'ungrouped'), ('hyperopt_splits-D-test-dddG', 'grouped')])

ESM-MSR_1
ESM-MSR_2
ESM-MSR_3
ESM-MSR_WT_1
ESM-MSR_WT_2
ESM-MSR_WT_3
ESM-MSR_MT_1
ESM-MSR_MT_2
ESM-MSR_MT_3
ESM-MSR_alpha_high_1
ESM-MSR_alpha_high_2
ESM-MSR_alpha_high_3
ESM-MSR_alpha_med_1
ESM-MSR_alpha_med_2
ESM-MSR_alpha_med_3
ESM-MSR_alpha_low_1
ESM-MSR_alpha_low_2
ESM-MSR_alpha_low_3
ESM-MSR_single_1
ESM-MSR_single_2
ESM-MSR_single_3
ESM-MSR_reg_1
ESM-MSR_reg_2
ESM-MSR_reg_3
ESM-MSR_defchain_1
ESM-MSR_defchain_2
ESM-MSR_defchain_3
ESM-MSR_defmarginal_1
ESM-MSR_defmarginal_2
ESM-MSR_defmarginal_3
ESM-MSR_masked_1
ESM-MSR_masked_2
ESM-MSR_masked_3
ESM-MSR_chain_1
ESM-MSR_chain_2
ESM-MSR_chain_3
ESM-MSR_noseqhead_1
ESM-MSR_noseqhead_2
ESM-MSR_noseqhead_3
ESM-MSR_noqkv_1
ESM-MSR_noqkv_2
ESM-MSR_noqkv_3
ESM-MSR_all_1
ESM-MSR_all_2
ESM-MSR_all_3
ESM-MSR_ffn_1
ESM-MSR_ffn_2
ESM-MSR_ffn_3
ESM-MSR_qkv_outproj_1
ESM-MSR_qkv_outproj_2
ESM-MSR_qkv_outproj_3
ESM-MSR_qkv_only_1
ESM-MSR_qkv_only_2
ESM-MSR_qkv_only_3
ESM-MSR_medmed_1
ESM-MSR_medmed_2
ESM-MSR_medmed_3
ESM-MSR_detach_reg_1
ESM-MSR

In [25]:
df_val = df_ddg_v.join(df_dddg_v)
df_val_display = collapse_replicates(df_val)
df_val_display

hyperopt_splits-S-val                 \
                                           ungrouped        grouped   
ESM-MSR                                0.787 ± 0.003  0.823 ± 0.002   
ESM-MSR_MT                             0.773 ± 0.003  0.817 ± 0.002   
ESM-MSR_WT                             0.772 ± 0.004  0.800 ± 0.002   
ESM-MSR_WT_additive                    0.772 ± 0.004  0.800 ± 0.002   
ESM-MSR_all                            0.786 ± 0.004  0.823 ± 0.001   
ESM-MSR_alpha_high                     0.790 ± 0.002  0.826 ± 0.002   
ESM-MSR_alpha_low                      0.608 ± 0.002  0.663 ± 0.003   
ESM-MSR_alpha_med                      0.714 ± 0.003  0.755 ± 0.003   
ESM-MSR_chain                          0.770 ± 0.003  0.818 ± 0.001   
ESM-MSR_defchain                       0.764 ± 0.004  0.810 ± 0.002   
ESM-MSR_defmarginal                    0.764 ± 0.004  0.810 ± 0.002   
ESM-MSR_detach_reg                     0.782 ± 0.001  0.822 ± 0.002   
ESM-MSR_ffn                            0.782 ± 0.005  0.819 ± 0.002   
ESM-MSR_fused                          0.770 ± 0.000  0.812 ± 0.001   
ESM-MSR_masked                         0.771 ± 0.005  0.814 ± 0.003   
ESM-MSR_medmed                         0.782 ± 0.003  0.820 ± 0.000   
ESM-MSR_noqkv                          0.777 ± 0.001  0.818 ± 0.002   
ESM-MSR_noseqhead                      0.788 ± 0.003  0.821 ± 0.001   
ESM-MSR_qkv_only                       0.771 ± 0.003  0.819 ± 0.001   
ESM-MSR_qkv_outproj                    0.779 ± 0.002  0.824 ± 0.001   
ESM-MSR_r1                             0.770 ± 0.000  0.812 ± 0.001   
ESM-MSR_reg                            0.787 ± 0.003  0.812 ± 0.001   
ESM-MSR_single                         0.788 ± 0.003  0.823 ± 0.002   
ESM3-large                                  0.430593       0.475533   
ESM3-medium                                 0.475441       0.542106   
ESM3-small                                  0.512754       0.558366   
ESM3-small-open                             0.486504       0.524120   
ESM3-small-open_chain                       0.508639       0.545133   
ESM3-small-open_masked                      0.508639       0.545133   
MutateEverything                       0.718 ± 0.005  0.729 ± 0.003   
MutateEverything_additive              0.732 ± 0.009  0.741 ± 0.008   
ProteinMPNN                                 0.563474       0.583753   
ProteinMPNN_masked                          0.563800       0.584209   
Rosetta Cartesian DDG                  0.628 ± 0.001  0.634 ± 0.001   
Rosetta Cartesian DDG_additive         0.628 ± 0.001  0.634 ± 0.001   
SPURS                                  0.598 ± 0.014  0.600 ± 0.017   
SPURS_additive                         0.687 ± 0.005  0.698 ± 0.006   
SPURS_reported                              0.740043       0.748886   
SPURS_reported_additive                     0.850746       0.864522   
ThermoMPNN(-D)                         0.731 ± 0.002  0.757 ± 0.002   
ThermoMPNN(-D)_additive                0.731 ± 0.002  0.757 ± 0.002   

                               hyperopt_splits-D-val                 \
                                           ungrouped        grouped   
ESM-MSR                                0.681 ± 0.017  0.702 ± 0.002   
ESM-MSR_MT                             0.527 ± 0.016  0.667 ± 0.008   
ESM-MSR_WT                             0.674 ± 0.019  0.614 ± 0.005   
ESM-MSR_WT_additive                    0.674 ± 0.019  0.614 ± 0.005   
ESM-MSR_all                            0.691 ± 0.019  0.700 ± 0.004   
ESM-MSR_alpha_high                     0.676 ± 0.020  0.712 ± 0.002   
ESM-MSR_alpha_low                      0.478 ± 0.001  0.468 ± 0.004   
ESM-MSR_alpha_med                      0.593 ± 0.002  0.602 ± 0.003   
ESM-MSR_chain                          0.646 ± 0.004  0.704 ± 0.006   
ESM-MSR_defchain                       0.645 ± 0.018  0.689 ± 0.003   
ESM-MSR_defmarginal                    0.622 ± 0.013  0.645 ± 0.001   
ESM-MSR_detach_reg                     0.679 ± 0.003  

In [17]:
df_test = df_ddg_t.join(df_dddg_t)
df_test_display = collapse_replicates(df_test)
df_test_display

hyperopt_splits-S-test                 \
                                            ungrouped        grouped   
ESM-MSR                                 0.774 ± 0.007  0.816 ± 0.002   
ESM-MSR_MT                              0.752 ± 0.006  0.804 ± 0.002   
ESM-MSR_WT                              0.770 ± 0.009  0.806 ± 0.005   
ESM-MSR_WT_additive                     0.770 ± 0.009  0.806 ± 0.005   
ESM-MSR_all                             0.775 ± 0.004  0.815 ± 0.003   
ESM-MSR_alpha_high                      0.763 ± 0.008  0.809 ± 0.003   
ESM-MSR_alpha_low                       0.693 ± 0.003  0.715 ± 0.004   
ESM-MSR_alpha_med                       0.758 ± 0.003  0.786 ± 0.004   
ESM-MSR_chain                           0.775 ± 0.002  0.814 ± 0.001   
ESM-MSR_defchain                        0.766 ± 0.009  0.810 ± 0.003   
ESM-MSR_defmarginal                     0.766 ± 0.009  0.810 ± 0.003   
ESM-MSR_detach_reg                      0.767 ± 0.003  0.815 ± 0.002   
ESM-MSR_ffn                             0.769 ± 0.002  0.813 ± 0.001   
ESM-MSR_fused                           0.763 ± 0.005  0.805 ± 0.005   
ESM-MSR_masked                          0.769 ± 0.001  0.807 ± 0.002   
ESM-MSR_medmed                          0.773 ± 0.003  0.815 ± 0.001   
ESM-MSR_noqkv                           0.775 ± 0.001  0.813 ± 0.001   
ESM-MSR_noseqhead                       0.768 ± 0.002  0.813 ± 0.002   
ESM-MSR_qkv_only                        0.768 ± 0.005  0.814 ± 0.001   
ESM-MSR_qkv_outproj                     0.772 ± 0.005  0.813 ± 0.002   
ESM-MSR_r1                              0.763 ± 0.005  0.805 ± 0.005   
ESM-MSR_reg                             0.773 ± 0.004  0.806 ± 0.003   
ESM-MSR_single                          0.764 ± 0.001  0.812 ± 0.001   
ESM3-large                                   0.538818       0.555275   
ESM3-medium                                  0.598086       0.600103   
ESM3-small                                   0.598344       0.613974   
ESM3-small-open                              0.545975       0.561370   
ESM3-small-open_chain                        0.559327       0.579499   
ESM3-small-open_masked                       0.559327       0.579499   
MutateEverything                        0.686 ± 0.009  0.693 ± 0.009   
MutateEverything_additive               0.696 ± 0.015  0.703 ± 0.017   
ProteinMPNN                                  0.545901       0.539303   
ProteinMPNN_masked                           0.546311       0.539719   
Rosetta Cartesian DDG                   0.621 ± 0.001  0.645 ± 0.001   
Rosetta Cartesian DDG_additive          0.621 ± 0.001  0.645 ± 0.001   
SPURS                                   0.606 ± 0.013  0.626 ± 0.017   
SPURS_additive                          0.680 ± 0.014  0.703 ± 0.013   
SPURS_reported                               0.720227       0.731902   
SPURS_reported_additive                      0.796582       0.805241   
ThermoMPNN(-D)                          0.729 ± 0.003  0.745 ± 0.003   
ThermoMPNN(-D)_additive                 0.729 ± 0.003  0.745 ± 0.003   

                               hyperopt_splits-D-test                 \
                                            ungrouped        grouped   
ESM-MSR                                 0.357 ± 0.007  0.798 ± 0.003   
ESM-MSR_MT                              0.319 ± 0.026  0.767 ± 0.010   
ESM-MSR_WT                              0.339 ± 0.022  0.726 ± 0.006   
ESM-MSR_WT_additive                     0.339 ± 0.022  0.726 ± 0.006   
ESM-MSR_all                             0.403 ± 0.034  0.806 ± 0.007   
ESM-MSR_alpha_high                      0.343 ± 0.009  0.791 ± 0.006   
ESM-MSR_alpha_low                       0.242 ± 0.008  0.596 ± 0.009   
ESM-MSR_alpha_med                       0.343 ± 0.013  0.729 ± 0.011   
ESM-MSR_chain                           0.331 ± 0.037  0.781 ± 0.015   
ESM-MSR_defchain                        0.278 ± 0.007  0.774 ± 0.006   
ESM-MSR_defmarginal                     0.348 ± 0.005  0.771 ± 0.005  

In [19]:
df_domainome = df_ddg_domainome
df_domainome_display = collapse_replicates(df_domainome)
df_domainome_display

domainome               
                               ungrouped        grouped
ESM-MSR                    0.546 ± 0.002  0.549 ± 0.003
ESM-MSR_MT                 0.524 ± 0.001  0.530 ± 0.001
ESM-MSR_WT                 0.545 ± 0.003  0.548 ± 0.004
ESM-MSR_all                0.538 ± 0.003  0.542 ± 0.004
ESM-MSR_alpha_high         0.525 ± 0.002  0.527 ± 0.004
ESM-MSR_alpha_low          0.550 ± 0.001  0.568 ± 0.001
ESM-MSR_alpha_med          0.564 ± 0.001  0.573 ± 0.001
ESM-MSR_chain              0.541 ± 0.004  0.550 ± 0.005
ESM-MSR_defchain           0.556 ± 0.002  0.565 ± 0.003
ESM-MSR_defmarginal        0.556 ± 0.002  0.565 ± 0.003
ESM-MSR_detach_reg         0.541 ± 0.008  0.544 ± 0.009
ESM-MSR_ffn                0.541 ± 0.004  0.543 ± 0.004
ESM-MSR_fused              0.528 ± 0.005  0.534 ± 0.006
ESM-MSR_masked             0.534 ± 0.007  0.539 ± 0.007
ESM-MSR_medmed             0.542 ± 0.004  0.546 ± 0.003
ESM-MSR_noqkv              0.536 ± 0.004  0.539 ± 0.005
ESM-MSR_noseqhead          0.544 ± 0.002  0.547 ± 0.002
ESM-MSR_qkv_only           0.534 ± 0.002  0.540 ± 0.004
ESM-MSR_qkv_outproj        0.534 ± 0.005  0.539 ± 0.006
ESM-MSR_r1                 0.528 ± 0.005  0.534 ± 0.006
ESM-MSR_reg                0.536 ± 0.005  0.540 ± 0.007
ESM-MSR_single             0.548 ± 0.003  0.549 ± 0.002
ESM3-large                      0.455502       0.507536
ESM3-medium                     0.503326       0.531377
ESM3-small                      0.528891       0.547119
ESM3-small-open                 0.508028       0.532973
ESM3-small-open_chain           0.526909       0.549980
ESM3-small-open_masked          0.526909       0.549980
MutateEverything           0.523 ± 0.001  0.517 ± 0.001
MutateEverything_additive  0.529 ± 0.000  0.524 ± 0.001
ProteinMPNN                     0.530813       0.547454
ProteinMPNN_masked              0.530908       0.547649
SPURS                      0.303 ± 0.029  0.279 ± 0.035
SPURS_additive             0.402 ± 0.005  0.376 ± 0.009
SPURS_reported                  0.411508       0.406233
SPURS_reported_additive         0.514502       0.513283
ThermoMPNN                 0.467 ± 0.005  0.455 ± 0.007

In [26]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Data Processing Function
# ==========================================
def process_data(df):
    """
    Groups replicates by index suffix (e.g., _1, _2), 
    calculates Mean and Std, and returns a formatted string DataFrame.
    """
    # Prevent silent processing of empty DataFrames
    assert not df.empty, "Input DataFrame 'df' is empty. Cannot process data."
    
    df = df.copy()
    
    # 1. Handle Index: Remove suffixes like _1, _2
    if df.index.name is None:
        # Assume first column is index if not set
        df.index.name = "ID" 
    
    # regex to strip _\d+ from the end
    base_names = df.index.astype(str).str.replace(r'_\d+$', '', regex=True)

    # 2. Group and Aggregate
    grouped = df.groupby(base_names)
    means = grouped.mean()
    stds = grouped.std()

    # 3. Format as "Mean $\pm$ Std"
    # We keep the structure of the means dataframe
    formatted_df = pd.DataFrame(index=means.index, columns=means.columns)
    
    for col in means.columns:
        m_col = means[col]
        s_col = stds[col]
        
        formatted_list = []
        for m, s in zip(m_col, s_col):
            if pd.isna(s): 
                # Single replicate case
                formatted_list.append(f"{m:.3f}")
            else:
                # Mean +/- Std (Using raw f-string to prevent invalid escape sequence warning)
                formatted_list.append(rf"{m:.3f} $\pm$ {s:.3f}")
                
        formatted_df[col] = formatted_list
        
    return formatted_df


# ==========================================
# 2. Custom LaTeX Generator
# ==========================================
def generate_latex_manual(df, title=r"Aggregated Results (Mean $\pm$ Std)"):
    """
    Manually constructs a LaTeX table to avoid Pandas to_latex pitfalls.
    Handles MultiIndex headers and special characters correctly.
    """
    # Prevent silent failure if passed an empty DataFrame
    assert not df.empty, "Input DataFrame 'df' is empty. Cannot generate LaTeX table."

    # Helper to escape LaTeX special characters
    def esc(text):
        if not isinstance(text, str): return str(text)
        # Replaces &, _, %, # with their escaped LaTeX equivalents
        text = text.replace('&', r'\&').replace('_', r'\_').replace('%', r'\%').replace('#', r'\#')
        # Replaces unicode epsilon with LaTeX math mode epsilon
        text = text.replace('ε', r'$\epsilon$')
        return text

    lines = []
    
    # --- A. Setup Table ---
    # Determine number of columns (Index + Data columns)
    n_cols = len(df.columns) + 1 
    col_format = "l" + "c" * (n_cols - 1)
    
    lines.append(r"\begin{table}[H]")
    lines.append(r"\centering")
    lines.append(r"\caption{" + title + r"}")
    lines.append(r"\label{tab:results}")
    lines.append(r"\resizebox{\textwidth}{!}{%") # Optional: Resize to fit page
    lines.append(r"\begin{tabular}{" + col_format + "}")
    lines.append(r"\toprule")

    # --- B. Generate Headers ---
    # We assume a 2-level MultiIndex for columns based on your example
    if isinstance(df.columns, pd.MultiIndex):
        levels = df.columns.levels
        codes = df.columns.codes
        
        # 1. Top Header Row (Handling Spans)
        top_row = []
        # Add empty cell for the Index column header
        top_row.append(r"{}") 
        
        # Iterate through the top level codes to find spans
        current_code = codes[0][0]
        span_count = 0
        
        for code in codes[0]:
            if code == current_code:
                span_count += 1
            else:
                # Append the previous group
                label = esc(levels[0][current_code])
                top_row.append(r"\multicolumn{" + str(span_count) + r"}{c}{" + label + r"}")
                # Reset
                current_code = code
                span_count = 1
        # Append final group
        label = esc(levels[0][current_code])
        top_row.append(r"\multicolumn{" + str(span_count) + r"}{c}{" + label + r"}")
        
        lines.append(" & ".join(top_row) + r" \\")
        
        # 2. CMidrules (Optional cosmetic lines)
        # Calculate positions for cmidrules matches the spans above
        cmid_cmds = []
        current_col_idx = 2 # Start at column 2 (1-based, skipping index)
        current_code = codes[0][0]
        span_count = 0
        
        for code in codes[0]:
            if code == current_code:
                span_count += 1
            else:
                cmid_cmds.append(r"\cmidrule(lr){" + f"{current_col_idx}-{current_col_idx+span_count-1}" + "}")
                current_col_idx += span_count
                current_code = code
                span_count = 1
        # Final cmidrule
        cmid_cmds.append(r"\cmidrule(lr){" + f"{current_col_idx}-{current_col_idx+span_count-1}" + "}")
        lines.append(" ".join(cmid_cmds))

        # 3. Second Header Row (Sub-groups)
        second_row = [esc(df.index.name) if df.index.name else "Group"]
        for code in codes[1]:
            second_row.append(esc(levels[1][code]))
        lines.append(" & ".join(second_row) + r" \\")

    else:
        # Simple flat header fallback
        headers = [esc(c) for c in df.columns]
        lines.append(" & ".join([r"{}"] + headers) + r" \\")

    lines.append(r"\midrule")

    # --- C. Generate Data Rows ---
    for idx, row in df.iterrows():
        # Escape the index name
        row_str = [esc(idx)]
        # Data values already contain math syntax ($\pm$), do NOT escape them.
        row_str.extend(row.astype(str).tolist())
        lines.append(" & ".join(row_str) + r" \\")

    # --- D. Close Table ---
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}") # End resizebox
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

In [27]:
df_val_out = process_data(df_val)
final_df = df_val_out
index = ['ESM-MSR', #'ESM-MSR_WT_additive',
         'ESM-MSR_WT', 'ESM-MSR_MT', 'ESM-MSR_defchain', 'ESM-MSR_defmarginal', 
         'ESM-MSR_alpha_high', 'ESM-MSR_alpha_med', 'ESM-MSR_alpha_low', 
         'ESM-MSR_chain', 'ESM-MSR_masked', 
         'ESM-MSR_single', 'ESM-MSR_reg', 'ESM-MSR_detach_reg', 'ESM-MSR_noseqhead', 
         'ESM-MSR_all', 'ESM-MSR_qkv_outproj', 'ESM-MSR_noqkv', 'ESM-MSR_ffn', 'ESM-MSR_qkv_only', 
         'ESM-MSR_r1', 'ESM-MSR_medmed', 'ESM-MSR_fused',
         'ESM3-small-open', 'ESM3-small-open_chain', 'ESM3-small-open_masked', 'ESM3-small', 'ESM3-medium', 'ESM3-large',
         'ProteinMPNN', #'ProteinMPNN_masked',
         'MutateEverything', 'MutateEverything_additive', 
         'ThermoMPNN(-D)', 'ThermoMPNN(-D)_additive', #'ThermoMPNN', 
         'SPURS', 'SPURS_additive', 'SPURS_reported', 'SPURS_reported_additive',
         'Rosetta Cartesian DDG', 'Rosetta Cartesian DDG_additive']
final_df = final_df.loc[index]
final_df.rename({
                'ESM-MSR': 'ESM-MSR (as reported)',
                'ESM-MSR_WT': 'ESM-MSR, WT LoRA only',
                'ESM-MSR_MT': 'ESM-MSR, MT LoRA only',
                'ESM-MSR_defchain': 'ESM-MSR, indep. masking',
                'ESM-MSR_defmarginal': 'ESM-MSR, masked marginal',
                'ESM-MSR_alpha_high': 'ESM-MSR, ε=1.25',
                'ESM-MSR_alpha_med': 'ESM-MSR, ε=0.5',
                'ESM-MSR_alpha_low': 'ESM-MSR, ε=0.25',
                'ESM-MSR_chain': 'ESM-MSR*, indep. masking',
                'ESM-MSR_masked': 'ESM-MSR*, masked marginal',
                'ESM-MSR_single': 'ESM-MSR*, singles only', 
                'ESM-MSR_reg': 'ESM-MSR*, no rank loss', 
                'ESM-MSR_noseqhead': 'ESM-MSR*, exclude seq. head',
                'ESM-MSR_all': 'ESM-MSR*, target QKV, FFN, out_proj.',
                'ESM-MSR_qkv_outproj': 'ESM-MSR*, target QKV and out_proj.',
                'ESM-MSR_noqkv': 'ESM-MSR*, target FFN and out_proj.', 
                'ESM-MSR_ffn': 'ESM-MSR*, only FFN up/down',
                'ESM-MSR_qkv_only': 'ESM-MSR*, only QKV proj.', 
                'ESM-MSR_r1': 'ESM-MSR*, WT&MT LoRA rank 1', 
                'ESM-MSR_medmed': 'ESM-MSR*, WT&MT LoRA rank 4',
                'ESM-MSR_detach_reg': 'ESM-MSR*, detach cal. head',
                'ESM-MSR_fused': 'ESM-MSR*, WT+MT LoRA shared',
                #'ESM-MSR_nobias': 'ESM-MSR*, no cal bias', 
                #'ESM-MSR_pool': 'ESM-MSR*, pool domains',
                'ESM3-small-open_chain': 'ESM3-small-open, indep. masking',
                'ESM3-small-open_masked': 'ESM3-small-open, masked marginal',
                'MutateEverything': 'Mutate Everything',
                'MutateEverything_additive': 'Mutate Everything, singles only\n add. approx.',
                'ThermoMPNN(-D)_additive': 'ThermoMPNN, additive approx.',
                'SPURS': 'SPURS-multi (retrained)',
                'SPURS_additive': 'SPURS (retrained), additive approx.',
                'SPURS_reported': 'SPURS-multi, HuggingFace model',
                'SPURS_reported_additive': 'SPURS, HuggingFace model add. approx.',
                'Rosetta Cartesian DDG_additive': 'Rosetta, add. approx.'
                }, inplace=True)
final_df
latex_code = generate_latex_manual(final_df, title="Ranking Results for Ablations and Variations: Validation Set")
print(latex_code)

\begin{table}[H]
\centering
\caption{Ranking Results for Ablations and Variations: Validation Set}
\label{tab:results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccccc}
\toprule
{} & \multicolumn{2}{c}{hyperopt\_splits-S-val} & \multicolumn{2}{c}{hyperopt\_splits-D-val} & \multicolumn{2}{c}{hyperopt\_splits-val} & \multicolumn{2}{c}{hyperopt\_splits-D-val-dddG} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
ID & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped \\
\midrule
ESM-MSR (as reported) & 0.787 $\pm$ 0.003 & 0.823 $\pm$ 0.002 & 0.681 $\pm$ 0.017 & 0.702 $\pm$ 0.002 & 0.819 $\pm$ 0.003 & 0.836 $\pm$ 0.001 & 0.524 $\pm$ 0.009 & 0.497 $\pm$ 0.012 \\
ESM-MSR, WT LoRA only & 0.772 $\pm$ 0.004 & 0.800 $\pm$ 0.002 & 0.674 $\pm$ 0.019 & 0.614 $\pm$ 0.005 & 0.809 $\pm$ 0.002 & 0.808 $\pm$ 0.002 & 0.062 $\pm$ 0.067 & 0.012 $\pm$ 0.006 \\
ESM-MSR, MT LoRA only & 0.773 $\pm$ 0.003 & 0.817 $\pm$ 0.002 & 0.527 $\pm$ 0.0

In [28]:
df_test_out = process_data(df_test)
final_df = df_test_out
index = ['ESM-MSR',
         'ESM-MSR_WT', 'ESM-MSR_MT', 'ESM-MSR_defchain', 'ESM-MSR_defmarginal',
         'ESM-MSR_alpha_high', 'ESM-MSR_alpha_med', 'ESM-MSR_alpha_low', 
         'ESM-MSR_chain', 'ESM-MSR_masked', 
         'ESM-MSR_single', 'ESM-MSR_reg', 'ESM-MSR_detach_reg', 'ESM-MSR_noseqhead', 
         'ESM-MSR_all', 'ESM-MSR_qkv_outproj', 'ESM-MSR_noqkv', 'ESM-MSR_ffn', 'ESM-MSR_qkv_only', 
         'ESM-MSR_r1', 'ESM-MSR_medmed', 'ESM-MSR_fused',
         'ESM3-small-open', 'ESM3-small-open_chain', 'ESM3-small-open_masked', 'ESM3-small', 'ESM3-medium', 'ESM3-large',
         'ProteinMPNN', #'ProteinMPNN_masked',
         'MutateEverything', 'MutateEverything_additive', 
         'ThermoMPNN(-D)', 'ThermoMPNN(-D)_additive', #'ThermoMPNN', 
         'SPURS', 'SPURS_additive', 'SPURS_reported', 'SPURS_reported_additive',
         'Rosetta Cartesian DDG', 'Rosetta Cartesian DDG_additive']
final_df = final_df.loc[index]
final_df.rename({
                'ESM-MSR': 'ESM-MSR (as reported)',
                'ESM-MSR_WT_additive': 'ESM-MSR, WT additive approx.',
                'ESM-MSR_WT': 'ESM-MSR, WT LoRA only',
                'ESM-MSR_MT': 'ESM-MSR, MT LoRA only',
                'ESM-MSR_defchain': 'ESM-MSR, indep. masking',
                'ESM-MSR_defmarginal': 'ESM-MSR, masked marginal',
                'ESM-MSR_alpha_high': 'ESM-MSR, ε=1.25',
                'ESM-MSR_alpha_med': 'ESM-MSR, ε=0.5',
                'ESM-MSR_alpha_low': 'ESM-MSR, ε=0.25',
                'ESM-MSR_chain': 'ESM-MSR*, indep. masking',
                'ESM-MSR_masked': 'ESM-MSR*, masked marginal',
                'ESM-MSR_single': 'ESM-MSR*, singles only', 
                'ESM-MSR_reg': 'ESM-MSR*, no rank loss', 
                'ESM-MSR_noseqhead': 'ESM-MSR*, exclude seq. head',
                'ESM-MSR_all': 'ESM-MSR*, target QKV, FFN, out_proj.',
                'ESM-MSR_qkv_outproj': 'ESM-MSR*, target QKV and out_proj.',
                'ESM-MSR_noqkv': 'ESM-MSR*, target FFN and out_proj.', 
                'ESM-MSR_ffn': 'ESM-MSR*, only FFN up/down',
                'ESM-MSR_qkv_only': 'ESM-MSR*, only QKV proj.', 
                'ESM-MSR_r1': 'ESM-MSR*, WT&MT LoRA rank 1', 
                'ESM-MSR_medmed': 'ESM-MSR*, WT&MT LoRA rank 4',
                'ESM-MSR_detach_reg': 'ESM-MSR*, detach cal. head',
                'ESM-MSR_fused': 'ESM-MSR*, WT+MT LoRA shared',
                #'ESM-MSR_qkv_only': 'ESM-MSR*, QKV only', 
                #'ESM-MSR_nobias': 'ESM-MSR*, no cal bias', 
                #'ESM-MSR_pool': 'ESM-MSR*, pool domains',
                'ESM3-small-open_chain': 'ESM-small-open, indep. masking',
                'ESM3-small-open_masked': 'ESM-small-open, masked marginal',
                'MutateEverything': 'Mutate Everything',
                'MutateEverything_additive': 'Mutate Everything, singles only\n additive approx.',
                'ThermoMPNN(-D)_additive': 'ThermoMPNN, additive approx.',
                'SPURS': 'SPURS-multi (retrained)',
                'SPURS_additive': 'SPURS (retrained), additive approx.',
                'SPURS_reported': 'SPURS-multi, HuggingFace model',
                'SPURS_reported_additive': 'SPURS, HuggingFace model add. approx.',
                'Rosetta Cartesian DDG_additive': 'Rosetta, add. approx.'
                }, inplace=True)
final_df
latex_code = generate_latex_manual(final_df, title="Ranking Results for Ablations and Variations: Test Set")
print(latex_code)

\begin{table}[H]
\centering
\caption{Ranking Results for Ablations and Variations: Test Set}
\label{tab:results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcccccccc}
\toprule
{} & \multicolumn{2}{c}{hyperopt\_splits-S-test} & \multicolumn{2}{c}{hyperopt\_splits-D-test} & \multicolumn{2}{c}{hyperopt\_splits-test} & \multicolumn{2}{c}{hyperopt\_splits-D-test-dddG} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
ID & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped & ungrouped & grouped \\
\midrule
ESM-MSR (as reported) & 0.774 $\pm$ 0.007 & 0.816 $\pm$ 0.002 & 0.357 $\pm$ 0.007 & 0.798 $\pm$ 0.003 & 0.727 $\pm$ 0.010 & 0.807 $\pm$ 0.003 & 0.263 $\pm$ 0.005 & 0.600 $\pm$ 0.005 \\
ESM-MSR, WT LoRA only & 0.770 $\pm$ 0.009 & 0.806 $\pm$ 0.005 & 0.339 $\pm$ 0.022 & 0.726 $\pm$ 0.006 & 0.762 $\pm$ 0.016 & 0.791 $\pm$ 0.005 & 0.029 $\pm$ 0.015 & 0.006 $\pm$ 0.062 \\
ESM-MSR, MT LoRA only & 0.752 $\pm$ 0.006 & 0.804 $\pm$ 0.002 & 0.319 $\pm$ 0.026

In [30]:
df_domainome_out = process_data(df_domainome)
final_df = df_domainome_out
index = ['ESM-MSR', #'ESM-MSR_additive',
         'ESM-MSR_WT', 'ESM-MSR_MT', 'ESM-MSR_defchain', 'ESM-MSR_defmarginal', 
         'ESM-MSR_alpha_high', 'ESM-MSR_alpha_med', 'ESM-MSR_alpha_low', 
         'ESM-MSR_chain', 'ESM-MSR_masked', 
         'ESM-MSR_single', 'ESM-MSR_reg', 'ESM-MSR_detach_reg', 'ESM-MSR_noseqhead', 
         'ESM-MSR_all', 'ESM-MSR_qkv_outproj', 'ESM-MSR_noqkv', 'ESM-MSR_ffn', 'ESM-MSR_qkv_only', 
         'ESM-MSR_r1', 'ESM-MSR_medmed', 'ESM-MSR_fused',
         'ESM3-small-open', 'ESM3-small-open_chain', 'ESM3-small-open_masked', 'ESM3-small', 'ESM3-medium', 'ESM3-large',
         'ProteinMPNN', #'ProteinMPNN_masked',
         'MutateEverything', #'MutateEverything_additive', 
         'ThermoMPNN', #'ThermoMPNN(-D)', 'ThermoMPNN(-D)_additive', 
         'SPURS', 'SPURS_additive', 'SPURS_reported', 'SPURS_reported_additive']
final_df = final_df.loc[index]
final_df.rename({
                'ESM-MSR': 'ESM-MSR (as reported)',
                'ESM-MSR_WT': 'ESM-MSR, WT LoRA only',
                'ESM-MSR_MT': 'ESM-MSR, MT LoRA only',
                'ESM-MSR_defchain': 'ESM-MSR, indep. masking',
                'ESM-MSR_defmarginal': 'ESM-MSR, masked marginal',
                'ESM-MSR_alpha_high': 'ESM-MSR, ε=1.25',
                'ESM-MSR_alpha_med': 'ESM-MSR, ε=0.5',
                'ESM-MSR_alpha_low': 'ESM-MSR, ε=0.25',
                'ESM-MSR_chain': 'ESM-MSR*, indep. masking',
                'ESM-MSR_masked': 'ESM-MSR*, masked marginal',
                'ESM-MSR_single': 'ESM-MSR*, singles only', 
                'ESM-MSR_reg': 'ESM-MSR*, no rank loss', 
                'ESM-MSR_noseqhead': 'ESM-MSR*, exclude seq. head',
                'ESM-MSR_all': 'ESM-MSR*, target QKV, FFN, out_proj.',
                'ESM-MSR_qkv_outproj': 'ESM-MSR*, target QKV and out_proj.',
                'ESM-MSR_noqkv': 'ESM-MSR*, target FFN and out_proj.', 
                'ESM-MSR_ffn': 'ESM-MSR*, only FFN up/down',
                'ESM-MSR_qkv_only': 'ESM-MSR*, only QKV proj.',  
                'ESM-MSR_r1': 'ESM-MSR*, WT&MT LoRA rank 1', 
                'ESM-MSR_medmed': 'ESM-MSR*, WT&MT LoRA rank 4',
                'ESM-MSR_detach_reg': 'ESM-MSR*, detach cal. head',
                'ESM-MSR_fused': 'ESM-MSR*, WT+MT LoRA shared',
                #'ESM-MSR_qkv_only': 'ESM-MSR*, QKV only', 
                #'ESM-MSR_nobias': 'ESM-MSR*, no cal bias', 
                #'ESM-MSR_pool': 'ESM-MSR*, pool domains',
                'ESM3-small-open_chain': 'ESM-small-open, indep. masking',
                'ESM3-small-open_masked': 'ESM-small-open, masked marginal',
                'MutateEverything': 'Mutate Everything',
                'MutateEverything_additive': 'Mutate Everything, singles only\n additive approx.',
                'ThermoMPNN(-D)_additive': 'ThermoMPNN, additive approx.',
                'SPURS': 'SPURS-multi (retrained)',
                'SPURS_additive': 'SPURS (retrained), additive approx.',
                'SPURS_reported': 'SPURS-multi, HuggingFace model',
                'SPURS_reported_additive': 'SPURS, HuggingFace model add. approx.'
                }, inplace=True)
final_df
latex_code = generate_latex_manual(final_df, title='Ranking Results for Ablations and Variations: Domainome')
print(latex_code)

\begin{table}[H]
\centering
\caption{Ranking Results for Ablations and Variations: Domainome}
\label{tab:results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{lcc}
\toprule
{} & \multicolumn{2}{c}{domainome} \\
\cmidrule(lr){2-3}
ID & ungrouped & grouped \\
\midrule
ESM-MSR (as reported) & 0.546 $\pm$ 0.002 & 0.549 $\pm$ 0.003 \\
ESM-MSR, WT LoRA only & 0.545 $\pm$ 0.003 & 0.548 $\pm$ 0.004 \\
ESM-MSR, MT LoRA only & 0.524 $\pm$ 0.001 & 0.530 $\pm$ 0.001 \\
ESM-MSR, indep. masking & 0.556 $\pm$ 0.002 & 0.565 $\pm$ 0.003 \\
ESM-MSR, masked marginal & 0.556 $\pm$ 0.002 & 0.565 $\pm$ 0.003 \\
ESM-MSR, $\epsilon$=1.25 & 0.525 $\pm$ 0.002 & 0.527 $\pm$ 0.004 \\
ESM-MSR, $\epsilon$=0.5 & 0.564 $\pm$ 0.001 & 0.573 $\pm$ 0.001 \\
ESM-MSR, $\epsilon$=0.25 & 0.550 $\pm$ 0.001 & 0.568 $\pm$ 0.001 \\
ESM-MSR*, indep. masking & 0.541 $\pm$ 0.004 & 0.550 $\pm$ 0.005 \\
ESM-MSR*, masked marginal & 0.534 $\pm$ 0.007 & 0.539 $\pm$ 0.007 \\
ESM-MSR*, singles only & 0.548 $\pm$ 0.003 & 0.549 $\pm$ 0.002 \